# V42 · Healthy purchase choices on the sugar panel

Estimate the switching household-year HMM and the fixed-class comparison on the same trips. The outcomes are category purchase incidence and conditional product choices. Spending enters the incidence covariate and descriptive wallet tables.

| Category | Code 1 | Code 2 | Code 3 |
|---|---|---|---|
| Fruit and vegetables | Fresh | Nonfresh | |
| Yogurt | Regular sugar | High sugar | |
| Cereal | Low sugar | Medium sugar | High sugar |
| Snack | Healthy | Unhealthy | |
| Beverage including milk | Healthy beverage or any milk | Less-healthy beverage without milk | |

Zero means no classified purchase in that category. The notebook reads the supplied yogurt code directly. The two yogurt spending columns describe category spending and appear in a coding audit. Milk qualifies at every fat level. If milk and a less-healthy beverage appear on the same trip, the combined choice is code 1. Cereal retains V39’s sequential gates.

The five incidence logits use food spending per trip divided by ten. Six conditional-choice logits use the state indicator, children, income, household size and age. The two switching logits use the six current-year life events. Each model estimates its own common initial probability $\pi$ by averaging the State 2 posterior in each household’s **first retained year**. Bootstrap refits use household multiplicities in that average. The 2007 fresh share is unused.

A panel with observed transitions has 61 switching-model parameters and 47 fixed-class parameters. With only singleton histories, both likelihoods reduce to the same 47-parameter mixture. State 2 is oriented by the fresh-FV gate. Review all category profiles for substantive interpretation.


## Run settings

Set the inclusive calendar bounds and minimum observed years per household below. The calendar window is applied first. A gap counts as one observed transition under the default policy. A filter with no eligible trips produces a sample-status file and finishes. A sample with only one observed year per household estimates the common mixture and omits transition estimates.

The four full-sample starts vary the initial probability, emission separation and switching persistence. Two starts can run concurrently. Each start runs to the same convergence criteria. The start log records the starting probability, final probability, likelihood and selection. Bootstrap draws warm-start at each model’s selected solution and re-estimate all free parameters. A failed fit gets one alternative initialization on the same sampled households.

Install dependencies if needed with `%pip install numpy pandas scipy duckdb pyarrow tqdm threadpoolctl numba` and restart the kernel.
Numba accelerates household recursions and count aggregation after its first compilation in a kernel. The notebook uses NumPy when Numba is unavailable or `USE_NUMBA = False`.


In [ ]:
import os

# Numerical-library thread settings MUST run before importing NumPy/SciPy.
# HMM_NUM_THREADS overrides the conservative default of min(4, detected CPUs).
_thread_text = os.environ.get("HMM_NUM_THREADS")
if _thread_text is None:
    HMM_NUM_THREADS = min(4, os.cpu_count() or 1)
else:
    try:
        HMM_NUM_THREADS = int(_thread_text)
    except (TypeError, ValueError) as exc:
        raise ValueError("HMM_NUM_THREADS must be an integer >= 1.") from exc
    if HMM_NUM_THREADS < 1:
        raise ValueError("HMM_NUM_THREADS must be an integer >= 1.")
for _thread_variable in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[_thread_variable] = str(HMM_NUM_THREADS)
print(f"Requested numerical threads = {HMM_NUM_THREADS}")
print("Thread settings take effect only before NumPy/SciPy import; after changing them, Restart Kernel and Run All.")

import gc
from copy import copy
import math
import time
import warnings
from collections import OrderedDict
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, Iterable, List, Mapping, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.special import expit, logsumexp
from scipy.stats import norm
try:
    from tqdm import tqdm
except ImportError:
    class tqdm:
        """Minimal progress fallback used only when tqdm is unavailable."""
        def __init__(self, iterable=None, total=None, desc="", **kwargs):
            self.iterable = iterable; self.total = total; self.desc = desc; self.n = 0
        def __enter__(self):
            print(self.desc); return self
        def __exit__(self, exc_type, exc, tb):
            return False
        def __iter__(self):
            for item in self.iterable:
                yield item; self.n += 1
        def update(self, value=1):
            self.n += value
        def set_description_str(self, text):
            self.desc = text; print(text)
        def set_postfix_str(self, text):
            pass
        def set_postfix(self, **kwargs):
            pass


import duckdb
import json
INPUT_PATH = Path(r"C:\Users\rwen13\Documents\healthy_purchase_data\processed\hmm_input\hmm_estimation_panel_sugar.parquet")
OUTPUT_DIR = INPUT_PATH.parent / 'V42_Sugar_Choice_Bootstrap_Results_Optimized'
USE_NUMBA = True  # Use compiled kernels when Numba is installed.
RUN_MODE = 'full_estimation'  # 'full_estimation', 'validate_only', or 'synthetic'
YEAR_MIN = 2008  # inclusive; None means open-ended
YEAR_MAX = 2017
N_STARTS = 4
START_WORKERS = 2  # Independent starts share read-only design/count arrays.
MAX_EM_ITERS = 150
SYNTHETIC_EM_ITERS = 40
MIN_OBSERVED_YEARS_PER_HOUSEHOLD = 3  # No extra three-year household filter.
YEAR_GAP_POLICY = 'observed_step'  # 'error' rejects calendar-year gaps.
DUCKDB_MEMORY_LIMIT = '8GB'
CHECK_DUPLICATE_TRIPS = True
TOL=1e-3
EM_REL_TOL=1e-10
EM_STABLE_ITERS=3
EM_PRINT_EVERY=5
OPT_FTOL=1e-14
OPT_GTOL=1e-8
OPT_NEWTON_STEPS=20
OPT_FAST_NEWTON_STEPS=8
SEED=20261030
CEREAL_CHOICE_WEIGHT=1.0
# The legacy initial_fresh_fv_share_2007 input column is ignored.
INITIAL_PI_START = 0.5
PI_EPS = 1e-10  # numerical boundary guard, not an informative prior
INCIDENCE_CACHE_RAM_MB = 4096  # construction-peak bound; larger caches stay on disk
BOOTSTRAP_WORKERS = 'auto'  # compares throughput with up to 4 workers using actual bootstrap draws
BOOTSTRAP_RESUME = True     # verifies source contents, numerical code and settings
BOOTSTRAP_EXPORT_EVERY = 10 # atomic per-draw checkpoints always save immediately
BOOTSTRAP_REPS = 200
BOOTSTRAP_SEED = 20261114
BOOTSTRAP_MIN_SUCCESS = 100
BOOTSTRAP_MIN_SUCCESS_FRACTION = 0.80
BOOTSTRAP_MAX_EM_ITERS = 300
SYNTHETIC_BOOTSTRAP_REPS = 5
BOOTSTRAP_BOUNDARY_TOL = 1e-7

TRANSITION_EVENT_COLUMNS=['event_new_child','getting_married','getting_divorce_or_separation','moving','getting_job','retirement']
TRANSITION_X_COLS=['intercept',*TRANSITION_EVENT_COLUMNS]
GATE_COVARIATE_COLS=['K_1','income10k_imputed','family_size_numeric','age']
GATE_X_COLS=['intercept','state_num',*GATE_COVARIATE_COLS]
CATEGORY_CONFIG=OrderedDict()
_spec=[
 ('fv',['fresh','nonfresh'],['fresh_spending','non_fresh_spending']),
 ('yogurt',['regular_sugar','high_sugar'],['regular_sugar_spending','high_sugar_spending']),
 ('cereal',['low_sugar','medium_sugar','high_sugar'],['low_sugar_cereal_spending','medium_sugar_cereal_spending','high_sugar_cereal_spending']),
 ('snack',['healthy','unhealthy'],['healthy_snack_spending','unhealthy_snack_spending']),
 ('beverage',['healthier','less_healthy'],['healthy_beverage_spending','unhealthy_beverage_spending',
  'skimmed_milk_spending','low_fat_milk_spending','other_milk_spending'])]
for c,labels,spending in _spec:
 gates=[dict(key=c+'_'+labels[0],prefix=c+'_'+labels[0]+'_gate',eligible_codes=tuple(range(1,len(labels)+1)),target_code=1)]
 if c=='cereal': gates.append(dict(key='cereal_medium_given_notlow',prefix='cereal_medium_given_notlow_gate',eligible_codes=(2,3),target_code=2))
 CATEGORY_CONFIG[c]=dict(y_col=c+'_y',outcomes=OrderedDict(enumerate(['no_classified_purchase' if c=='cereal' else 'no_purchase',*labels])),gates=gates,spending=spending,choice_weight=1.0)
CATEGORY_NAMES=list(CATEGORY_CONFIG)
assert CATEGORY_NAMES == ['fv','yogurt','cereal','snack','beverage']
assert 'milk' not in CATEGORY_CONFIG
GATE_KEYS=[g['key'] for c in CATEGORY_CONFIG.values() for g in c['gates']]
np.random.seed(SEED)


Requested numerical threads = 4
Thread settings take effect only before NumPy/SciPy import; after changing them, Restart Kernel and Run All.


## 1. Numerical engine
Analytic gradients and exact small-block Hessians support damped Newton steps with L-BFGS fallback. Scaling affects only optimizer coordinates. The objective has no ridge penalty. Original gradient/EM tolerances, stable-iteration requirements, and downhill-step rejection are retained. Hessians are used for numerical optimization and rank diagnostics only; reported inference uses household bootstrap variances.


In [2]:
# Optional Numba kernels use float64 and release the GIL across independent fits.
try:
    import numba
    from numba import njit
except ImportError:
    numba = None
    def njit(**options):
        return lambda function: function


@njit(nogil=True)
def _gate_counts(design, annual, success, failure, weight, gamma, choice_weight, n_design):
    counts = np.zeros((4, n_design))
    for i in range(len(annual)):
        h, d = annual[i], design[i]
        if weight[h] == 0:
            continue
        w1 = gamma[h, 0] * choice_weight * weight[h]
        w2 = gamma[h, 1] * choice_weight * weight[h]
        counts[0, d] += w1 * success[i]
        counts[1, d] += w1 * failure[i]
        counts[2, d] += w2 * success[i]
        counts[3, d] += w2 * failure[i]
    return counts


@njit(nogil=True)
def _incidence_counts(codes, household, counts, weights, n_spend):
    # Scan this potentially multi-GB cache once; no row-sized per-draw temporaries.
    totals = np.zeros((counts.shape[1], n_spend))
    for i in range(len(codes)):
        w = weights[household[i]]
        if w:
            for j in range(counts.shape[1]):
                totals[j, codes[i]] += w * counts[i, j]
    return totals


@njit(nogil=True)
def _chain_posteriors(starts, ends, emission, pattern, log_trans, log_pi):
    """Two-state log-space forward/backward in contiguous household order."""
    gamma = np.empty_like(emission)
    xi = np.empty((len(emission) - len(starts), 4))
    household_ll = np.empty(len(starts))
    # The forward messages live in gamma until consumed by the backward pass.
    for h in range(len(starts)):
        first, last = starts[h], ends[h] - 1
        gamma[first, 0] = log_pi[0] + emission[first, 0]
        gamma[first, 1] = log_pi[1] + emission[first, 1]
        for t in range(first + 1, last + 1):
            trans = log_trans[pattern[t]]
            a, b = gamma[t-1, 0], gamma[t-1, 1]
            gamma[t, 0] = emission[t, 0] + np.logaddexp(a + trans[0], b + trans[2])
            gamma[t, 1] = emission[t, 1] + np.logaddexp(a + trans[1], b + trans[3])
        ll = np.logaddexp(gamma[last, 0], gamma[last, 1])
        household_ll[h] = ll
        b0, b1 = 0., 0.
        for t in range(last, first - 1, -1):
            p0 = np.exp(gamma[t, 0] + b0 - ll)
            p1 = np.exp(gamma[t, 1] + b1 - ll)
            gamma[t, 0], gamma[t, 1] = p0 / (p0 + p1), p1 / (p0 + p1)
            if t == first:
                break
            trans = log_trans[pattern[t]]
            a, b = gamma[t-1, 0], gamma[t-1, 1]
            e0, e1 = emission[t, 0], emission[t, 1]
            # Each preceding household contributes one annual row without a transition.
            row = t - h - 1
            x00 = np.exp(a + trans[0] + e0 + b0 - ll)
            x01 = np.exp(a + trans[1] + e1 + b1 - ll)
            x10 = np.exp(b + trans[2] + e0 + b0 - ll)
            x11 = np.exp(b + trans[3] + e1 + b1 - ll)
            total = x00 + x01 + x10 + x11
            xi[row, 0], xi[row, 1] = x00 / total, x01 / total
            xi[row, 2], xi[row, 3] = x10 / total, x11 / total
            n0, n1 = e0 + b0, e1 + b1
            b0 = np.logaddexp(trans[0] + n0, trans[1] + n1)
            b1 = np.logaddexp(trans[2] + n0, trans[3] + n1)
    return household_ll, gamma, xi


@dataclass
class GateBlock:
    """Annual outcome counts mapped to one shared unique-covariate design."""

    key: str
    category: str
    prefix: str
    x_base: np.ndarray
    target: np.ndarray
    hy_pos: np.ndarray
    eligible_rows: int
    choice_weight: float
    success_count: np.ndarray
    failure_count: np.ndarray
    design_pos: np.ndarray


@dataclass
class ModelArrays:
    """NumPy model data with cached household-year and gate structures."""

    purchase: Dict[str, np.ndarray]
    outcome: Dict[str, np.ndarray]
    spend: Dict[str, np.ndarray]
    food_total_spend: np.ndarray
    food_total_spend_10: np.ndarray
    gate_covariates: Dict[str, np.ndarray]
    gate_blocks: Dict[str, GateBlock]
    trip_hy_pos: np.ndarray
    household_year: pd.DataFrame
    hy_key: pd.MultiIndex
    hh_blocks: Dict[str, Dict[str, np.ndarray]]
    households: List[str]
    years_by_household: Dict[str, List[int]]
    hh_starts: np.ndarray
    hh_ends: np.ndarray
    transition_x: np.ndarray
    transition_curr_pos: np.ndarray
    transition_row_of_hy: np.ndarray


@dataclass
class PreparedData:
    """Annual sufficient statistics and model buffers.

    With RETAIN_TRIP_DATA=False, df is a count-only RangeIndex shell carrying
    the trip count and sorting audit. Raw trip buffers are released after the
    incidence fits; all subsequent EM operations use annual/design counts.
    """

    df: pd.DataFrame
    hy: pd.DataFrame
    arrays: ModelArrays
    annual_stats: pd.DataFrame


@dataclass
class EstimationResult:
    """Fitted model blocks and final optimizer diagnostics."""

    prepared: PreparedData
    incidence_models: Dict[str, Any]
    gate_params: Dict[str, Any]
    beta_from1: np.ndarray
    beta_from2: np.ndarray
    pi_state2: float
    gamma_df: pd.DataFrame
    history_df: pd.DataFrame
    initial_state_table: pd.DataFrame
    transition_table: pd.DataFrame
    log_likelihood: float
    transition_fits: Optional[Dict[str, Dict[str, Any]]] = None
    optimizer_diagnostics_df: Optional[pd.DataFrame] = None


def safe_log(x: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    """Take a stable logarithm of probabilities."""
    return np.log(np.clip(x, eps, 1.0))


def sigmoid(x: np.ndarray) -> np.ndarray:
    """Stable logistic transform."""
    return expit(x)


def significance_stars(p_value: float) -> str:
    """Return conventional significance stars."""
    if not np.isfinite(p_value):
        return ""
    if p_value < 0.01:
        return "***"
    if p_value < 0.05:
        return "**"
    if p_value < 0.10:
        return "*"
    return ""


def ensure_columns(df: pd.DataFrame, required_cols: Sequence[str]) -> None:
    """Raise a clear error when required columns are missing."""
    missing = [col for col in required_cols if col not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")


def first_nonmissing(series: pd.Series) -> float:
    """Return the first observed value, preserving missingness when all values are missing."""
    observed = series.dropna()
    return observed.iloc[0] if len(observed) else np.nan


def _optimizer_diagnostics(
    result: Any,
    block_name: str,
    objective_value: float,
    gradient: np.ndarray,
    n_observations: int,
    effective_weight: float,
    covariance_success: bool,
    solver: str,
) -> Dict[str, Any]:
    return {
        "scaled_gradient_inf": float(getattr(result, 'scaled_gradient_inf', np.nan)),
        "initial_objective": float(getattr(result, 'initial_objective', np.nan)),
        "objective_improvement": float(getattr(result, 'objective_improvement', np.nan)),
        "reverted_to_start": bool(getattr(result, 'reverted_to_start', False)),
        "lbfgs_message": str(getattr(result, 'lbfgs_message', '')),
        "newton_refinements": int(getattr(result, 'newton_refinements', 0)),
        "model_block": block_name,
        "solver": solver,
        "optimizer_success": bool(result.success),
        "status": int(result.status),
        "message": str(result.message),
        "iterations": int(getattr(result, "nit", -1)),
        "function_evaluations": int(getattr(result, "nfev", -1)),
        "gradient_evaluations": int(getattr(result, "njev", -1)),
        "final_objective": float(objective_value),
        "gradient_norm": float(np.linalg.norm(gradient)),
        "n_observations": int(n_observations),
        "effective_weight": float(effective_weight),
        "covariance_hessian_success": bool(covariance_success),
    }


def _finish_logit_fit(
    result: Any,
    hessian_function: Any,
    gradient_function: Any,
    param_names: Sequence[str],
    block_name: str,
    objective_function: Any,
    n_observations: int,
    effective_weight: float,
    compute_se: bool,
    solver: str,
) -> Dict[str, Any]:
    beta = np.asarray(result.x, dtype=np.float64)
    # Block Hessians are used for numerical optimization/rank diagnostics only.
    # No inverse-Hessian variance path is permitted, even when called directly.
    if compute_se:
        raise ValueError('Block-Hessian inference is disabled. Run the household bootstrap for variance and SEs.')
    covariance_success = False
    hessian = np.full((len(beta), len(beta)), np.nan)
    covariance = np.full((len(beta), len(beta)), np.nan)
    gradient = np.asarray(gradient_function(beta), dtype=np.float64)
    objective_value = float(objective_function(beta))
    table = pd.DataFrame({"parameter": list(param_names), "estimate": beta})
    diagnostics = _optimizer_diagnostics(
        result, block_name, objective_value, gradient, n_observations,
        effective_weight, covariance_success, solver,
    )
    if not result.success and not getattr(result,'evaluation_only',False):
        warnings.warn(
            f"Optimizer did not report success for {block_name}: "
            f"status={result.status}; message={result.message}",
            RuntimeWarning,
        )
    return {
        "_hessian_function": hessian_function,
        "result": result,
        "coef": beta,
        "hessian": hessian,
        "cov": covariance,
        "table": table,
        "nll": objective_value,
        "diagnostics": diagnostics,
    }


def _cached_value_gradient(callback):
    """Reuse one matrix product/sigmoid pass for both L-BFGS-B callbacks."""
    last_beta = None
    last_result = None
    def evaluate(beta):
        nonlocal last_beta, last_result
        if last_beta is None or not np.array_equal(beta, last_beta):
            last_beta = np.array(beta, copy=True)
            last_result = callback(beta)
        return last_result
    return evaluate


def _minimize_logit(evaluate, hess, initial, effective_weight, maxiter, zero_diagonal=None):
    """Scaled L-BFGS with gradient verification and damped Newton refinement.

    Scaling changes only optimizer coordinates, never data or likelihood.
    The starting coefficients are always a candidate, so a block M-step
    cannot decrease its expected complete-data log likelihood.
    """
    from scipy.optimize import OptimizeResult
    weight = max(float(effective_weight), 1.0)
    initial = np.array(initial, dtype=float, copy=True)
    if zero_diagonal is None:
        zero_diagonal = np.diag(hess(np.zeros_like(initial)))
    curvature = np.maximum(zero_diagonal / weight, 0.0)
    scale = np.sqrt(curvature)
    scale[scale < 1e-8] = 1.0
    calls = 0

    def objective(theta):
        nonlocal calls
        calls += 1
        value, gradient = evaluate(theta / scale)
        return value / weight, gradient / (weight * scale)

    theta0 = initial * scale
    initial_value, _ = evaluate(initial)
    f0, g0 = objective(theta0)
    best_theta, best_f = theta0.copy(), f0
    raw_message = 'Newton-first step; L-BFGS fallback not required'
    nit = 0
    # Six-/seven-parameter convex blocks: a few exact Newton steps normally
    # reach the same gradient tolerance much sooner than L-BFGS warm starts.
    # Damping, monotone line search and L-BFGS fallback preserve safeguards.
    fast_refinements = 0
    for _ in range(min(OPT_FAST_NEWTON_STEPS, maxiter)):
        value, grad = objective(best_theta)
        if np.linalg.norm(grad, ord=np.inf) <= OPT_GTOL:
            break
        H = hess(best_theta / scale) / weight / scale[:, None] / scale[None, :]
        jitter = 1e-10 * max(1.0, float(np.max(np.diag(H))))
        try:
            direction = -np.linalg.solve(H + jitter*np.eye(len(H)), grad)
        except np.linalg.LinAlgError:
            direction = -np.linalg.pinv(H) @ grad
        slope = float(grad @ direction)
        if not np.isfinite(direction).all() or slope >= 0:
            break
        accepted = False
        for backtrack in range(30):
            theta = best_theta + (0.5**backtrack)*direction
            candidate_f, _ = objective(theta)
            if np.isfinite(candidate_f) and candidate_f <= value + 1e-4*(0.5**backtrack)*slope:
                best_theta, best_f = theta, candidate_f
                fast_refinements += 1
                accepted = True
                break
        if not accepted:
            break
    _, fast_gradient = objective(best_theta)
    if maxiter > 0 and np.linalg.norm(fast_gradient, ord=np.inf) > OPT_GTOL:
        res = minimize(objective, best_theta, jac=True, method='L-BFGS-B', options={
            'maxiter': maxiter, 'maxfun': max(20000, 30 * maxiter),
            'ftol': OPT_FTOL, 'gtol': OPT_GTOL, 'maxls': 50, 'maxcor': 20,
        })
        raw_message, nit = str(res.message), int(res.nit)
        candidate_f, _ = objective(res.x)
        if np.isfinite(candidate_f) and candidate_f <= best_f:
            best_theta, best_f = res.x.copy(), candidate_f

    # A relative-function stop alone is insufficient on a massive likelihood.
    # Small dense Hessians (one logit block) make Newton polishing affordable.
    refinements = 0
    for _ in range(OPT_NEWTON_STEPS if maxiter > 0 else 0):
        value, grad = objective(best_theta)
        if np.linalg.norm(grad, ord=np.inf) <= OPT_GTOL:
            break
        H = hess(best_theta / scale) / weight / scale[:, None] / scale[None, :]
        jitter = 1e-10 * max(1.0, float(np.max(np.diag(H))))
        try:
            direction = -np.linalg.solve(H + jitter * np.eye(len(H)), grad)
        except np.linalg.LinAlgError:
            direction = -np.linalg.pinv(H) @ grad
        slope = float(grad @ direction)
        if not np.isfinite(direction).all() or slope >= 0:
            direction, slope = -grad, -float(grad @ grad)
        accepted = False
        for backtrack in range(40):
            step = 0.5 ** backtrack
            theta = best_theta + step * direction
            candidate_f, _ = objective(theta)
            if np.isfinite(candidate_f) and candidate_f <= value + 1e-4 * step * slope:
                best_theta, best_f = theta, candidate_f
                accepted = True
                refinements += 1
                break
        if not accepted:
            break
    # Recheck unscaled loss, preventing a rounding artifact in division from
    # accepting an objectively worse solution than the warm start.
    beta = best_theta / scale
    final_value, final_gradient = evaluate(beta)
    reverted = not np.isfinite(final_value) or final_value > initial_value
    if reverted:
        beta = initial
        final_value, final_gradient = evaluate(beta)
    scaled_gradient = final_gradient / (weight * scale)
    score = float(np.linalg.norm(scaled_gradient, ord=np.inf))
    success = bool(np.isfinite(score) and score <= OPT_GTOL)
    return OptimizeResult(
        x=beta, fun=final_value, jac=final_gradient, success=success, evaluation_only=(maxiter==0),
        status=0 if success else 2, nit=nit + refinements + fast_refinements, nfev=calls, njev=calls,
        message=('Scaled gradient tolerance met' if success else
                 'Scaled gradient tolerance NOT met; inspect this block'),
        scaled_gradient_inf=score, initial_objective=initial_value,
        objective_improvement=initial_value-final_value, reverted_to_start=reverted,
        lbfgs_message=raw_message, newton_refinements=refinements + fast_refinements,
    )




def fit_weighted_logit(
    X, y, sample_weight=None, start=None, param_names=None, maxiter=300,
    compute_se=False, block_name="weighted_logit",
):
    """Weighted Bernoulli likelihood, evaluated once per optimizer candidate."""
    y = np.asarray(y, dtype=np.float64)
    weight = np.ones_like(y) if sample_weight is None else np.asarray(sample_weight, dtype=np.float64)
    if np.any(np.isfinite(weight) & (weight < 0)):
        raise ValueError(f"{block_name}: negative weights are not allowed.")
    names = list(param_names) if param_names is not None else [f"b{i}" for i in range(X.shape[1])]
    return fit_binomial_count_logit(
        X, weight*y, weight*(1-y), start, names, block_name, maxiter, compute_se,
    )


def fit_compressed_gate_logit(
    X_base, y, weight_state1, weight_state2, start, param_names, block_name,
    maxiter=300, compute_se=False,
):
    """Compatibility wrapper for the original two-state weighted-trip API."""
    y = np.asarray(y, dtype=np.float64)
    w1 = np.asarray(weight_state1, dtype=np.float64)
    w2 = np.asarray(weight_state2, dtype=np.float64)
    return fit_gate_count_logit(
        X_base, w1*y, w1*(1-y), w2*y, w2*(1-y), start,
        param_names, block_name, maxiter, compute_se,
    )


def fit_gate_count_logit(
    X_base, success1, failure1, success2, failure2, start,
    param_names, block_name, maxiter=300, compute_se=False,
):
    """Exact two-state binomial likelihood on unique, unrounded covariates.

    Counts already include annual posterior weights and the category weight.
    No binomial combinatorial constant is added (the original trip likelihood).
    """
    X_base = np.asarray(X_base, dtype=np.float64)
    counts = np.column_stack([success1, failure1, success2, failure2]).astype(np.float64, copy=False)
    if np.any(np.isfinite(counts) & (counts < 0)):
        raise ValueError(f"{block_name}: negative expected counts are not allowed.")
    keep = np.isfinite(counts).all(axis=1) & (counts.sum(axis=1)>0) & np.isfinite(X_base).all(axis=1)
    if not keep.all():
        X_base, counts = X_base[keep], counts[keep]
    if not len(counts):
        raise ValueError(f"{block_name}: no valid observations remain.")
    s1, f1, s2, f2 = counts.T
    t1, t2 = s1+f1, s2+f2
    p_count = X_base.shape[1]+1
    base_positions = np.r_[0, np.arange(2, p_count)]
    initial = np.zeros(p_count) if start is None else np.asarray(start, dtype=np.float64)

    component_beta = None
    component_values = None
    def components(beta):
        nonlocal component_beta, component_values
        if component_beta is None or not np.array_equal(beta, component_beta):
            shared = X_base @ beta[base_positions]
            component_values = (shared, sigmoid(shared+beta[1]), sigmoid(shared+2*beta[1]))
            component_beta = np.array(beta, copy=True)
        return component_values

    def value_gradient(beta):
        shared, p1, p2 = components(beta)
        eta1, eta2 = shared + beta[1], shared + 2 * beta[1]
        # Stable Bernoulli loss, exactly consistent with the analytic gradient.
        value = (np.dot(s1, np.maximum(-eta1, 0)) + np.dot(f1, np.maximum(eta1, 0))
                 + np.dot(t1, np.log1p(np.exp(-np.abs(eta1))))
                 + np.dot(s2, np.maximum(-eta2, 0)) + np.dot(f2, np.maximum(eta2, 0))
                 + np.dot(t2, np.log1p(np.exp(-np.abs(eta2)))))
        r1, r2 = t1*p1-s1, t2*p2-s2
        out = np.empty(p_count)
        out[base_positions] = X_base.T @ (r1+r2)
        out[1] = np.sum(r1+2*r2)
        return float(value), out
    evaluate = _cached_value_gradient(value_gradient)

    def hess(beta):
        _, p1, p2 = components(beta)
        c1, c2 = t1*p1*(1-p1), t2*p2*(1-p2)
        out = np.empty((p_count, p_count))
        out[np.ix_(base_positions, base_positions)] = X_base.T @ (X_base*(c1+c2)[:, None])
        cross = X_base.T @ (c1+2*c2)
        out[base_positions, 1] = cross
        out[1, base_positions] = cross
        out[1, 1] = np.sum(c1+4*c2)
        return out

    zero_diagonal = np.empty(p_count)
    zero_diagonal[base_positions] = 0.25 * np.einsum('i,ij,ij->j', t1+t2, X_base, X_base)
    zero_diagonal[1] = 0.25 * np.sum(t1 + 4*t2)
    result = _minimize_logit(evaluate, hess, initial, float(counts.sum()), maxiter, zero_diagonal)
    return _finish_logit_fit(
        result, hess, lambda b:evaluate(b)[1], param_names, block_name,
        lambda b:evaluate(b)[0], len(counts), float(counts.sum()), compute_se,
        "scaled_lbfgs_newton_gate_logit",
    )


def fit_binomial_count_logit(
    X, success, failure, start, param_names, block_name, maxiter=300, compute_se=False,
):
    """Original binomial-count objective with a joint loss/gradient callback."""
    X = np.asarray(X, dtype=np.float64)
    success, failure = np.asarray(success, dtype=np.float64), np.asarray(failure, dtype=np.float64)
    if np.any(np.isfinite(success) & (success<0)) or np.any(np.isfinite(failure) & (failure<0)):
        raise ValueError(f"{block_name}: negative expected counts are not allowed.")
    total = success+failure
    keep = np.isfinite(success) & np.isfinite(failure) & (total>0) & np.isfinite(X).all(axis=1)
    if not keep.all():
        X, success, failure, total = X[keep], success[keep], failure[keep], total[keep]
    if not len(total):
        raise ValueError(f"{block_name}: no positive expected counts remain.")
    initial = np.zeros(X.shape[1]) if start is None else np.asarray(start, dtype=np.float64)

    def value_gradient(beta):
        eta = X @ beta
        p = sigmoid(eta)
        value = np.sum(success*np.logaddexp(0, -eta) + failure*np.logaddexp(0, eta))
        return float(value), X.T @ (total*p-success)
    evaluate = _cached_value_gradient(value_gradient)
    def hess(beta):
        p = sigmoid(X @ beta)
        return X.T @ (X*(total*p*(1-p))[:, None])

    zero_diagonal = 0.25 * np.einsum('i,ij,ij->j', total, X, X)
    result = _minimize_logit(evaluate, hess, initial, float(total.sum()), maxiter, zero_diagonal)
    return _finish_logit_fit(
        result, hess, lambda b:evaluate(b)[1], param_names, block_name,
        lambda b:evaluate(b)[0], len(total), float(total.sum()), compute_se,
        "scaled_lbfgs_newton_binomial_logit",
    )


def finalize_diagnostics(fit):
    """Record block rank for diagnostics; bootstrap supplies all reported SEs."""
    if '_hessian_function' not in fit:
        return  # Already diagnosed; shared incidence fits are finalized once.
    hessian = np.asarray(fit.pop('_hessian_function')(fit['coef']), dtype=float)
    fit['diagnostics']['hessian_rank'] = int(np.linalg.matrix_rank(hessian))
    fit['diagnostics']['parameter_count'] = len(fit['coef'])
    fit['diagnostics']['covariance_hessian_success'] = False


def set_household_weights(prepared, weights=None):
    """Integer multiplicities represent resampling complete household sequences."""
    arrays = prepared.arrays
    H = len(arrays.hh_starts)
    w = np.ones(H, dtype=float) if weights is None else np.asarray(weights, dtype=float)
    if w.shape != (H,) or not np.isfinite(w).all() or (w < 0).any() or not np.equal(w, np.floor(w)).all() or w.sum() != H:
        raise ValueError('Require H nonnegative integer household multiplicities summing to H.')
    arrays._hh_weights = w.copy()
    arrays._hy_weights = np.repeat(w, arrays.hh_ends-arrays.hh_starts)




## 2. Parquet validation and exact sufficient statistics
DuckDB validates the same original input and builds exact annual choice counts. A one-time household-by-unrounded-spending cache also supports full incidence refits in every bootstrap draw. Caches with a construction peak above `INCIDENCE_CACHE_RAM_MB` stay on disk. State-independent incidence cancels from state posteriors and is added once to the full likelihood. Trips are never expanded by latent state in Python.


In [3]:
def sql_name(name):
    return '"'+name.replace('"','""')+'"'


class EmptySample(ValueError):
    """The requested sample cannot be formed from the selected observations."""


def sample_tag():
    for bound in (YEAR_MIN, YEAR_MAX):
        if bound is not None and type(bound) is not int:
            raise ValueError('YEAR_MIN and YEAR_MAX must be integers or None.')
    if YEAR_MIN is not None and YEAR_MAX is not None and YEAR_MIN > YEAR_MAX:
        raise ValueError('YEAR_MIN must be <= YEAR_MAX.')
    if type(MIN_OBSERVED_YEARS_PER_HOUSEHOLD) is not int or MIN_OBSERVED_YEARS_PER_HOUSEHOLD < 1:
        raise ValueError('Minimum observed years must be a positive integer.')
    return f"years_{YEAR_MIN or 'open'}_{YEAR_MAX or 'open'}_minHH{MIN_OBSERVED_YEARS_PER_HOUSEHOLD}_{YEAR_GAP_POLICY}"


def register_combined_beverage_view(con, path):
    """Read sugar-panel outcomes; merge only in a SQL view, never edit Parquet.

    Existing milk category is assumed to contain ordinary milk only, as agreed
    for this first test. No UPC/flavor screening or sweetened-milk relabeling.
    Validate BOTH original codes before replacement so milk cannot mask an
    invalid/missing beverage value. Nulls are errors, never non-purchases.
    """
    source = con.from_parquet(str(path))
    if 'panel_year' not in source.columns:
        raise ValueError('Missing panel_year in input parquet.')
    if YEAR_MIN is not None:
        source = source.filter(f'panel_year >= {int(YEAR_MIN)}')
    if YEAR_MAX is not None:
        source = source.filter(f'panel_year <= {int(YEAR_MAX)}')
    source.create_view('raw_source')
    actual={r[0] for r in con.execute('DESCRIBE raw_source').fetchall()}
    missing={'milk_y','beverage_y'}-actual
    if missing:
        con.close()
        raise ValueError(f'Missing source outcome columns: {missing}')
    invalid=con.execute("""
        SELECT count(*) FROM raw_source
        WHERE milk_y IS NULL OR beverage_y IS NULL
           OR NOT isfinite(CAST(milk_y AS DOUBLE))
           OR NOT isfinite(CAST(beverage_y AS DOUBLE))
           OR milk_y NOT IN (0,1,2) OR beverage_y NOT IN (0,1,2)
    """).fetchone()[0]
    if invalid:
        con.close()
        raise ValueError(f'{invalid:,} invalid original milk/beverage outcome rows; fix upstream.')
    con.execute("""
        CREATE VIEW raw AS SELECT * REPLACE (
            CASE WHEN milk_y IN (1,2) OR beverage_y = 1 THEN 1
                 WHEN beverage_y = 2 THEN 2 ELSE 0 END AS beverage_y
        ) FROM raw_source
    """)


def prepare_parquet(path, scratch_dir):
    """Validate first, then aggregate; never silently drop malformed records."""
    con=duckdb.connect()
    try:
        return _prepare_parquet(con, path, scratch_dir)
    except BaseException:
        con.close()
        raise


def _prepare_parquet(con, path, scratch_dir):
    con.execute(f"SET threads={HMM_NUM_THREADS}")
    con.execute(f"SET memory_limit='{DUCKDB_MEMORY_LIMIT}'")
    scratch_dir=Path(scratch_dir); scratch_dir.mkdir(parents=True,exist_ok=True)
    con.execute("SET temp_directory='"+str(scratch_dir).replace("'","''")+"'")
    con.execute("SET enable_progress_bar=false")
    register_combined_beverage_view(con, path)
    controls=[*GATE_COVARIATE_COLS,*TRANSITION_EVENT_COLUMNS]
    spends=[s for cfg in CATEGORY_CONFIG.values() for s in cfg['spending']]
    required=['hhid','panel_year','trip_id','purchase_date','food_spend',*controls,*[c+'_y' for c in CATEGORY_NAMES],*spends]
    actual={r[0] for r in con.execute('DESCRIBE raw').fetchall()}
    if set(required)-actual: raise ValueError(f'Missing columns: {set(required)-actual}')
    bad=[f'{sql_name(c)} IS NULL' for c in required]
    numeric=['panel_year','food_spend',*controls,*spends,*[c+'_y' for c in CATEGORY_NAMES]]
    bad += [f'NOT isfinite(CAST({sql_name(c)} AS DOUBLE))' for c in numeric]
    bad += ['panel_year != floor(panel_year)',"TRY_CAST(purchase_date AS DATE) IS NULL"]
    bad += [f'{c} NOT IN (0,1)' for c in [*TRANSITION_EVENT_COLUMNS,'K_1']]
    bad += [f'{c}_y NOT IN ({",".join(map(str,cfg["outcomes"]))})' for c,cfg in CATEGORY_CONFIG.items()]
    bad += ['family_size_numeric <= 0','age <= 0','age > 120','income10k_imputed < 0']
    n,b=con.execute('SELECT count(*), count(*) FILTER (WHERE '+ ' OR '.join(bad)+') FROM raw').fetchone()
    if b: raise ValueError(f'{b:,} invalid rows; fix upstream rather than silently filtering.')
    if not n: raise EmptySample('No trips in the calendar window.')
    if CHECK_DUPLICATE_TRIPS:
        duplicate=con.execute('SELECT 1 FROM raw GROUP BY hhid,panel_year,trip_id HAVING count(*)>1 LIMIT 1').fetchone()
        if duplicate: raise ValueError('Duplicate household-year-trip key.')
    # SQL aliases retain the internal V29 household key to reuse its tested recursion.
    aggs=['CAST(hhid AS VARCHAR) AS household_code','CAST(panel_year AS INTEGER) AS panel_year','count(*) AS n_trips',
          'sum(CAST(food_spend AS DOUBLE)) AS food_total_spend_sum',
          'min(CAST(purchase_date AS DATE)) AS first_purchase_date','max(CAST(purchase_date AS DATE)) AS last_purchase_date']
    aggs += [f'first(CAST({c} AS DOUBLE)) AS {c}' for c in controls]
    for c,cfg in CATEGORY_CONFIG.items():
        aggs += [f'sum(CAST({c}_y>0 AS BIGINT)) AS {c}_purchase_n',
                 'sum('+ '+'.join(f'CAST({v} AS DOUBLE)' for v in cfg['spending'])+f') AS {c}_spend_sum']
        aggs += [f'sum(CAST({c}_y={k} AS BIGINT)) AS {c}_{v}_n' for k,v in cfg['outcomes'].items() if k]
    checks = ' OR '.join(f'min({c}) != max({c})' for c in controls)
    # The annual table supplies the validation, eligibility and EM counts in one scan.
    con.execute('CREATE TEMP TABLE annual_input AS SELECT hhid, ' + ','.join(aggs) +
                ', (' + checks + ') AS inconsistent FROM raw GROUP BY hhid,panel_year')
    if con.execute('SELECT 1 FROM annual_input WHERE inconsistent LIMIT 1').fetchone():
        raise ValueError('Demographics/events vary within a household-year.')
    if MIN_OBSERVED_YEARS_PER_HOUSEHOLD == 1:
        con.execute('CREATE VIEW trips AS SELECT * FROM raw')
        annual_from = 'annual_input'
    else:
        con.execute(f'CREATE TEMP TABLE eligible AS SELECT hhid FROM annual_input GROUP BY hhid HAVING count(*) >= {MIN_OBSERVED_YEARS_PER_HOUSEHOLD}')
        con.execute('CREATE VIEW trips AS SELECT r.* FROM raw r SEMI JOIN eligible e USING(hhid)')
        annual_from = 'annual_input SEMI JOIN eligible USING(hhid)'
    hy = con.execute('SELECT * EXCLUDE (hhid, inconsistent) FROM ' + annual_from +
                     ' ORDER BY household_code,panel_year').df()
    con.execute('DROP TABLE annual_input')
    # Only nine combinations: cheap audit of the original codes and merged codes.
    audit_source = "raw_source r" if MIN_OBSERVED_YEARS_PER_HOUSEHOLD == 1 else "raw_source r SEMI JOIN eligible e USING(hhid)"
    merge_audit=con.execute(f"""
        SELECT r.milk_y AS original_milk_y, r.beverage_y AS original_beverage_y,
               CASE WHEN r.milk_y IN (1,2) OR r.beverage_y=1 THEN 1
                    WHEN r.beverage_y=2 THEN 2 ELSE 0 END AS merged_beverage_y,
               count(*) AS n_trips
        FROM {audit_source}
        GROUP BY 1,2,3 ORDER BY 1,2
    """).df()
    if hy.empty: raise EmptySample('No households pass the minimum observed-years filter.')
    hh=hy.household_code.to_numpy(); starts=np.flatnonzero(np.r_[True,hh[1:]!=hh[:-1]])
    ends=np.r_[starts[1:],len(hy)]
    gaps=(hy.groupby('household_code',sort=False).panel_year.diff()>1)
    if YEAR_GAP_POLICY not in ('observed_step','error'): raise ValueError('Invalid YEAR_GAP_POLICY')
    if gaps.any() and YEAR_GAP_POLICY=='error': raise ValueError(f'{gaps.sum()} observed-year gaps; explicit handling required.')
    hy['chain_step']=hy.groupby('household_code',sort=False).cumcount()+1
    x=np.column_stack([np.ones(len(hy)),hy[TRANSITION_EVENT_COLUMNS].to_numpy(float)])
    mask=np.ones(len(hy),bool); mask[starts]=False
    curr=np.flatnonzero(mask); row=np.full(len(hy),-1,dtype=np.int64); row[curr]=np.arange(len(curr))
    annual_cols=['household_code','panel_year','n_trips','food_total_spend_sum',*TRANSITION_EVENT_COLUMNS]
    annual_cols += [c for c in hy if c.endswith(('_n','_spend_sum')) and c not in annual_cols]
    annual=hy[annual_cols].copy()
    base=np.column_stack([np.ones(len(hy)),hy[GATE_COVARIATE_COLS].to_numpy(float)])
    unique,mapping=np.unique(base,axis=0,return_inverse=True)
    blocks={}
    for c,cfg in CATEGORY_CONFIG.items():
        for gate in cfg['gates']:
            success=hy[f'{c}_{cfg["outcomes"][gate["target_code"]]}_n'].to_numpy(float)
            total=sum(hy[f'{c}_{cfg["outcomes"][v]}_n'].to_numpy(float) for v in gate['eligible_codes'])
            active=np.flatnonzero(total>0)
            if not len(active): raise EmptySample(f'No eligible choices to estimate gate {gate["key"]}')
            blocks[gate['key']]=GateBlock(gate['key'],c,gate['prefix'],unique,success[active]/total[active],active,int(total.sum()),1.,success[active],total[active]-success[active],mapping[active])
    arrays=ModelArrays({}, {}, {}, np.array([]),np.array([]),{},blocks,np.array([],dtype=int),hy,
        pd.MultiIndex.from_frame(hy[['household_code','panel_year']]),{},hh[starts].tolist(),{},starts,ends,x,curr,row)
    flags = x[:, 1:].astype(np.int64) @ (1 << np.arange(len(TRANSITION_EVENT_COLUMNS)))
    present, annual_codes = np.unique(flags, return_inverse=True)
    design = np.column_stack([np.ones(len(present)), (present[:, None] >> np.arange(len(TRANSITION_EVENT_COLUMNS))) & 1])
    arrays._annual_event_patterns = design, annual_codes
    arrays._connection=con
    arrays._annual_statistics=annual
    shell=pd.DataFrame(index=pd.RangeIndex(int(hy.n_trips.sum())))
    shell.attrs['beverage_merge_audit']=merge_audit
    shell.attrs['yogurt_code_audit']=con.execute('''
        SELECT yogurt_y, regular_sugar_spending > 0 AS regular_spend_positive,
               high_sugar_spending > 0 AS high_spend_positive, count(*) AS n_trips
        FROM trips GROUP BY 1,2,3 ORDER BY 1,2,3
    ''').df()
    shell.attrs['sorting_audit']={'calendar_year_min':YEAR_MIN, 'calendar_year_max':YEAR_MAX, 'trips_in_calendar_window':n, 'input_trips':n,'retained_trips':len(shell),'minimum_observed_years_required':MIN_OBSERVED_YEARS_PER_HOUSEHOLD,
        'trips_removed_by_minimum_years_filter':n-len(shell),'observed_year_gaps':int(gaps.sum()),'year_gap_policy':YEAR_GAP_POLICY}
    print(f'Input {n:,} trips; retained {len(shell):,}; {len(starts):,} households; {len(hy):,} household-years; {gaps.sum():,} gaps.')
    prepared = PreparedData(shell,hy,arrays,annual)
    prepared._source_path=str(Path(path).resolve())
    set_household_weights(prepared)
    prepare_incidence_cache(prepared, scratch_dir)
    return prepared


def prepare_incidence_cache(prepared, scratch_dir):
    """Cache exact household x spending counts once; never round spending.

    State-independent incidence factors out of the state recursion. Its full
    refitted log likelihood is added once to the sequence likelihood. A bounded
    in-memory cache avoids all trip scans in bootstrap; large caches stay on disk.
    """
    arrays=prepared.arrays; con=arrays._connection
    path=Path(scratch_dir)/'incidence_household_spend.parquet'
    household_index=pd.DataFrame({'household_code':arrays.households,
                                 'household_index':np.arange(len(arrays.households),dtype=np.int64)})
    con.register('incidence_household_index',household_index)
    fields=['h.household_index','CAST(food_spend AS DOUBLE)/10.0 AS spend10','count(*) AS n']
    fields += [f'sum(CAST({c}_y>0 AS BIGINT)) AS {c}' for c in CATEGORY_NAMES]
    sql='SELECT '+','.join(fields)+' FROM trips JOIN incidence_household_index h ON CAST(trips.hhid AS VARCHAR)=h.household_code GROUP BY 1,2'
    path_sql="'"+str(path).replace("'","''")+"'"
    con.execute('COPY ('+sql+') TO '+path_sql+" (FORMAT PARQUET, COMPRESSION ZSTD)")
    con.unregister('incidence_household_index')
    rows=int(con.execute('SELECT count(*) FROM read_parquet('+path_sql+')').fetchone()[0])
    arrays._incidence_cache_path=str(path.resolve())
    arrays._incidence_cache=None
    # Bound construction peak as well as steady-state storage; no RAM guessing.
    estimated_peak_bytes=rows*160
    if estimated_peak_bytes<=INCIDENCE_CACHE_RAM_MB*1024**2:
        columns=con.execute('SELECT * FROM read_parquet('+path_sql+')').fetchnumpy()
        spend,codes=np.unique(columns.pop('spend10'),return_inverse=True)
        index=columns.pop('household_index')
        if USE_NUMBA and numba is not None:
            # n bounds every category count. Choose a lossless unsigned width.
            names = ['n', *CATEGORY_NAMES]
            dtype = np.min_scalar_type(int(columns['n'].max()))
            packed = np.empty((rows, len(names)), dtype=dtype)
            for j, name in enumerate(names):
                packed[:, j] = columns.pop(name)
            columns = packed
        arrays._incidence_cache=(spend,codes,index,columns)
        cache_mode='memory'
    else:
        cache_mode='disk'
    arrays._incidence_cache_mode=cache_mode
    prepared.df.attrs['incidence_cache']={'mode':cache_mode,'rows':rows,'memory_limit_mb':INCIDENCE_CACHE_RAM_MB}
    print(f'Exact incidence cache: {rows:,} household-spending groups; {cache_mode} mode.')


def fit_incidence_models(arrays, progress_bar=None):
    """Fully refit all incidence logits from exact household-weighted counts."""
    cache=arrays._incidence_cache
    if cache is not None:
        spend,codes,index,columns=cache
        names = ['n', *CATEGORY_NAMES]
        if isinstance(columns, np.ndarray):
            sums = _incidence_counts(codes, index, columns, arrays._hh_weights, len(spend))
            counts = dict(zip(names, sums))
        else:
            sample_weight = arrays._hh_weights[index]
            counts = {c: np.bincount(codes, weights=columns[c]*sample_weight, minlength=len(spend))
                      for c in names}
        keep=counts['n']>0
        spend=spend[keep];counts={c:values[keep] for c,values in counts.items()}
    else:
        # Each caller gets a private connection, also safe with bootstrap threads.
        con=duckdb.connect()
        con.execute('SET threads=1')
        con.execute("SET memory_limit='2GB'")
        try:
            weights=pd.DataFrame({'household_index':np.arange(len(arrays._hh_weights)),
                                  'bootstrap_weight':arrays._hh_weights})
            con.register('bootstrap_weights',weights)
            path_sql="'"+arrays._incidence_cache_path.replace("'","''")+"'"
            fields=['spend10',*[f'sum(d.{c}*w.bootstrap_weight) AS {c}' for c in ['n',*CATEGORY_NAMES]]]
            counts=con.execute('SELECT '+','.join(fields)+' FROM read_parquet('+path_sql+
                ') d JOIN bootstrap_weights w USING(household_index) WHERE bootstrap_weight>0 GROUP BY spend10 ORDER BY spend10').fetchnumpy()
            spend=counts.pop('spend10')
        finally:
            con.close()
    X=np.column_stack([np.ones(len(spend)),spend])
    starts=getattr(arrays,'_incidence_starts',{})
    fits={}
    for c in CATEGORY_NAMES:
        fits[c]=fit_binomial_count_logit(X,counts[c],counts['n']-counts[c],starts.get(c),
            [f'{c}_inc_intercept',f'{c}_inc_food_total_spend_10'],f'incidence:{c}')
    return fits


def initial_gate_weights(prepared, start_id=0):
    """Non-identical emission starts, independent of the initial pi starting value.

    Smoothed annual fresh share supplies a weak initial ordering. Its only role
    is numerical initialization; subsequent posterior weights come from the HMM.
    """
    hy=getattr(prepared, '_initialization_panel', prepared.hy)
    score=(hy.fv_fresh_n.to_numpy(float)+.5)/(hy.fv_purchase_n.to_numpy(float)+1.)
    z=(score-score.mean())/max(score.std(),1e-8)
    if start_id:
        rng = np.random.default_rng([SEED, start_id])
        weights = rng.uniform(.25, 1.5, len(CATEGORY_CONFIG))
        scores = []
        for category, cfg in CATEGORY_CONFIG.items():
            count = hy[f'{category}_{cfg["outcomes"][1]}_n'].to_numpy(float)
            total = hy[f'{category}_purchase_n'].to_numpy(float)
            share = (count + .5) / (total + 1.)
            scores.append((share - share.mean()) / max(share.std(), 1e-8))
        z = np.average(scores, axis=0, weights=weights) * rng.uniform(1.5, 3.0)
    g=expit(.7*np.clip(z,-5,5))
    if hasattr(prepared, '_initialization_rows'):
        g = g[prepared._initialization_rows]
    out=prepared.hy[['household_code','panel_year']].copy()
    out['gamma_state_1']=1-g; out['gamma_state_2']=g
    return out


## 3. E-step and safeguarded EM
Forward–backward runs in log space on contiguous household histories. The NumPy fallback batches histories of equal length. Posterior tables are built after EM finishes. Six choice regressions use posterior-weighted annual counts. Two transition regressions use at most 64 event patterns. Incidence is separable and refitted from cached exact counts once per bootstrap sample, then shared by its two models. Each model estimates its own common pi from first-observed-year posteriors, counting each household once. Bootstrap multiplicities enter the household likelihood, all M-step counts, and profile aggregates. Warm starts do not fix any parameters. EM convergence is local, not a global-optimum guarantee.


In [4]:
def start_probability(start_id):
    if start_id == 0:
        return INITIAL_PI_START
    return float(np.random.default_rng([SEED, start_id, 1]).uniform(.15, .85))


def _align_gamma_to_hy(gamma_df: pd.DataFrame, arrays: ModelArrays) -> Tuple[np.ndarray, np.ndarray]:
    """Use zero-copy aligned posterior fast path, retaining a safe keyed fallback."""
    if len(gamma_df) == len(arrays.household_year):
        same_hh = np.array_equal(
            gamma_df["household_code"].to_numpy(copy=False),
            arrays.household_year["household_code"].to_numpy(copy=False),
        )
        same_year = np.array_equal(
            gamma_df["panel_year"].to_numpy(dtype=np.int64),
            arrays.household_year["panel_year"].to_numpy(dtype=np.int64),
        )
        if same_hh and same_year:
            return (
                gamma_df["gamma_state_1"].to_numpy(dtype=np.float64, copy=False),
                gamma_df["gamma_state_2"].to_numpy(dtype=np.float64, copy=False),
            )
    gamma_key = pd.MultiIndex.from_frame(gamma_df[["household_code", "panel_year"]])
    aligned = gamma_df.set_index(gamma_key).reindex(arrays.hy_key)
    if aligned[["gamma_state_1", "gamma_state_2"]].isna().any().any():
        raise ValueError("gamma_df cannot be aligned to the household-year index.")
    return aligned["gamma_state_1"].to_numpy(dtype=np.float64), aligned["gamma_state_2"].to_numpy(dtype=np.float64)

def fit_gate_models(
    arrays: ModelArrays,
    gamma_df: pd.DataFrame,
    progress_bar: Optional[Any] = None,
    em_iteration: Optional[int] = None,
    max_em_iters: Optional[int] = None,
    previous_fits: Optional[Mapping[str, Any]] = None,
) -> Dict[str, Any]:
    """Fit cached compressed gate likelihoods with one weight pair per eligible row."""
    if isinstance(gamma_df, np.ndarray):
        gamma = gamma_df
    else:
        gamma = np.column_stack(_align_gamma_to_hy(gamma_df, arrays))
    gamma1, gamma2 = gamma.T
    fitted: Dict[str, Any] = {}
    for gate_key in GATE_KEYS:
        block = arrays.gate_blocks[gate_key]
        if progress_bar is not None:
            if em_iteration is None:
                progress_bar.set_description_str(f"Initialization: {progress_bar.n + 1}/{progress_bar.total} | gate={gate_key}")
            else:
                progress_bar.set_description_str(f"EM {em_iteration}/{max_em_iters} | gate={gate_key}")
        # Sum fractional successes/failures over annual groups with the SAME
        # exact design. Four small count vectors replace all trip-state rows.
        n_design = len(block.x_base)
        if USE_NUMBA and numba is not None:
            counts = _gate_counts(block.design_pos, block.hy_pos, block.success_count,
                                  block.failure_count, arrays._hy_weights, gamma,
                                  block.choice_weight, n_design)
        else:
            w1 = gamma1[block.hy_pos]*block.choice_weight*arrays._hy_weights[block.hy_pos]
            w2 = gamma2[block.hy_pos]*block.choice_weight*arrays._hy_weights[block.hy_pos]
            counts = [np.bincount(block.design_pos, weights=w*count, minlength=n_design)
                      for w,count in ((w1,block.success_count), (w1,block.failure_count),
                                      (w2,block.success_count), (w2,block.failure_count))]
        start = previous_fits[gate_key]["coef"] if previous_fits is not None else None
        fitted[gate_key] = fit_gate_count_logit(
            block.x_base, *counts, start=start,
            param_names=[f"{block.prefix}_{col}" for col in GATE_X_COLS],
            block_name=f"gate:{gate_key}",
        )
        fitted[gate_key]["diagnostics"]["compressed_design_rows"] = n_design
        fitted[gate_key]["diagnostics"]["n_observations"] = block.eligible_rows
        if progress_bar is not None and em_iteration is None:
            progress_bar.update(1)
    return fitted

def log_emission_by_household_year(arrays, incidence_models, gate_params):
    """Evaluate the original trip likelihood using exact annual gate counts."""
    n_hy = len(arrays.household_year)
    # Incidence is independent of state, so it cancels in every posterior.
    # Retain its exact weighted LL separately for fit statistics/EM safeguards.
    arrays._incidence_loglik = -sum(float(incidence_models[c]['nll']) for c in CATEGORY_NAMES)
    state1, state2 = np.zeros(n_hy), np.zeros(n_hy)
    for key in GATE_KEYS:
        block = arrays.gate_blocks[key]
        coef = np.asarray(gate_params[key]["coef"], dtype=np.float64)
        shared = block.x_base @ np.r_[coef[0], coef[2:]]
        for state_num, destination in ((1.0,state1), (2.0,state2)):
            eta = shared + state_num * coef[1]
            tail = np.log1p(np.exp(-np.abs(eta)))
            lp, lq = np.minimum(eta, 0) - tail, np.minimum(-eta, 0) - tail
            ll = (block.success_count*lp[block.design_pos]
                  +block.failure_count*lq[block.design_pos])
            # hy_pos is unique in a gate, so direct indexed addition is valid.
            destination[block.hy_pos] += block.choice_weight*ll
    return np.column_stack([state1,state2])

def posterior_frame(arrays, gamma_values):
    gamma_df = arrays.household_year[['household_code', 'panel_year']].copy()
    gamma_df['gamma_state_1'], gamma_df['gamma_state_2'] = gamma_values.T
    gamma_df['decoded_state'] = np.where(gamma_values[:, 1] > gamma_values[:, 0], 2, 1)
    return gamma_df


def forward_backward(arrays: ModelArrays, emission_loglik: np.ndarray, pi_state2: float,
                     beta_from1: np.ndarray, beta_from2: np.ndarray, arrays_only=False):
    """Same log-space recursions, vectorized over equal-length households.

    Observed-year gaps count as one step only under the explicit observed_step policy.
    """
    n_hy = len(arrays.household_year)
    design, pattern = arrays._annual_event_patterns
    eta1, eta2 = design @ beta_from1, design @ beta_from2
    pattern_logs = np.column_stack([-np.logaddexp(0, eta1), -np.logaddexp(0, -eta1),
                                    -np.logaddexp(0, eta2), -np.logaddexp(0, -eta2)])
    log_pi = np.array([safe_log(1-pi_state2), safe_log(pi_state2)])
    if USE_NUMBA and numba is not None:
        household_lls, gamma_values, xi_values = _chain_posteriors(
            arrays.hh_starts, arrays.hh_ends, emission_loglik, pattern, pattern_logs, log_pi)
    else:
        gamma_values = np.empty((n_hy, 2), dtype=np.float64)
        xi_values = np.empty((len(arrays.transition_curr_pos), 4), dtype=np.float64)
        log_trans = pattern_logs[pattern].reshape(n_hy, 2, 2)
        if not hasattr(arrays, '_sequence_batches'):
            lengths = arrays.hh_ends - arrays.hh_starts
            arrays._sequence_batches = [(np.flatnonzero(lengths == t),
                arrays.hh_starts[lengths == t, None] + np.arange(t)) for t in np.unique(lengths)]
        household_lls = np.empty(len(arrays.hh_starts))
        for hh_indices, pos in arrays._sequence_batches:
            em = emission_loglik[pos]
            trans = log_trans[pos]
            alpha = np.zeros_like(em); backward = np.zeros_like(em)
            alpha[:, 0] = log_pi + em[:, 0]
            for t in range(1, pos.shape[1]):
                alpha[:, t, 0] = em[:, t, 0]+np.logaddexp(alpha[:, t-1, 0]+trans[:, t, 0, 0], alpha[:, t-1, 1]+trans[:, t, 1, 0])
                alpha[:, t, 1] = em[:, t, 1]+np.logaddexp(alpha[:, t-1, 0]+trans[:, t, 0, 1], alpha[:, t-1, 1]+trans[:, t, 1, 1])
            ll = np.logaddexp(alpha[:, -1, 0], alpha[:, -1, 1])
            household_lls[hh_indices] = ll
            for t in range(pos.shape[1]-2, -1, -1):
                nxt = em[:, t+1]+backward[:, t+1]
                backward[:, t, 0] = np.logaddexp(trans[:, t+1, 0, 0]+nxt[:, 0], trans[:, t+1, 0, 1]+nxt[:, 1])
                backward[:, t, 1] = np.logaddexp(trans[:, t+1, 1, 0]+nxt[:, 0], trans[:, t+1, 1, 1]+nxt[:, 1])
            gamma = np.exp(alpha + backward - ll[:, None, None])
            gamma /= gamma.sum(axis=2, keepdims=True)
            gamma_values[pos] = gamma
            for t in range(1, pos.shape[1]):
                xi = np.exp(alpha[:, t-1, :, None] + trans[:, t] + em[:, t, None, :]
                            + backward[:, t, None, :] - ll[:, None, None])
                xi /= xi.sum(axis=(1, 2), keepdims=True)
                xi_values[arrays.transition_row_of_hy[pos[:, t]]] = xi.reshape(-1, 4)
    # Sum in original household order to minimize numerical changes.
    total_ll = float(arrays._hh_weights @ household_lls) + arrays._incidence_loglik
    if arrays_only:
        return total_ll, gamma_values, xi_values
    gamma_df = posterior_frame(arrays, gamma_values)
    curr = arrays.transition_curr_pos
    xi_df = pd.DataFrame({'household_code': arrays.household_year['household_code'].to_numpy()[curr],
        'year_prev': arrays.household_year['panel_year'].to_numpy()[curr-1],
        'year_curr': arrays.household_year['panel_year'].to_numpy()[curr],
        **{c: xi_values[:, j] for j, c in enumerate(['xi_11', 'xi_12', 'xi_21', 'xi_22'])},
        **{c: arrays.transition_x[curr, j] for j, c in enumerate(TRANSITION_EVENT_COLUMNS, 1)}})
    return total_ll, gamma_df, xi_df

def fit_transition_models(
    xi_df: pd.DataFrame,
    start_from1: Optional[np.ndarray] = None,
    start_from2: Optional[np.ndarray] = None,
    arrays: Optional[ModelArrays] = None,
    maxiter: int = 300,
) -> Tuple[Dict[str, Any], Dict[str, Any]]:
    """Fit both reference-preserving transition logits from compressed counts."""
    # Six binary flags imply at most 64 distinct designs. Cache their mapping.
    if arrays is not None and hasattr(arrays, "_transition_grouping"):
        X, codes = arrays._transition_grouping
    else:
        flags = (arrays.transition_x[arrays.transition_curr_pos, 1:].astype(np.int8) if arrays is not None
                 else np.column_stack([xi_df[c].to_numpy(dtype=np.int8) for c in TRANSITION_EVENT_COLUMNS]))
        all_codes = flags @ (1 << np.arange(len(TRANSITION_EVENT_COLUMNS)))
        present, codes = np.unique(all_codes, return_inverse=True)
        X = np.column_stack([np.ones(len(present)),
                             ((present[:, None] >> np.arange(len(TRANSITION_EVENT_COLUMNS))) & 1)])
        codes = codes.astype(np.int32)
        if arrays is not None:
            arrays._transition_grouping = X, codes
    names = ("xi_11", "xi_12", "xi_21", "xi_22")
    xi = xi_df if isinstance(xi_df, np.ndarray) else xi_df[list(names)].to_numpy()
    weights = arrays._hy_weights[arrays.transition_curr_pos] if arrays is not None else 1.0
    counts = {name: np.bincount(codes, weights=xi[:, j]*weights, minlength=len(X))
              for j, name in enumerate(names)}
    fit1 = fit_binomial_count_logit(
        X, counts["xi_12"], counts["xi_11"], start_from1,
        [f"trans_from1_{col}" for col in TRANSITION_X_COLS], "transition:from_state_1", maxiter=maxiter,
    )
    fit2 = fit_binomial_count_logit(
        X, counts["xi_22"], counts["xi_21"], start_from2,
        [f"trans_from2_{col}" for col in TRANSITION_X_COLS], "transition:from_state_2", maxiter=maxiter,
    )
    for fit in (fit1,fit2):
        fit["diagnostics"]["compressed_design_rows"] = len(X)
        fit["diagnostics"]["n_observations"] = len(xi_df)
    return fit1, fit2

def _run_monotone_em(prepared, incidence, max_em_iters, tolerance, no_switch=False, start_result=None, verbose=True, start_id=0):
    """Shared EM policy: warm starts, observed-LL guard, persistent plateau.

    A rejected downhill proposal is logged, never relabeled as an improvement.
    Roundoff-sized rejections can count toward a stable plateau only when all
    candidate M-steps satisfy their gradient checks. Larger drops stop with
    converged=False and preserve the last accepted coefficients/posteriors.
    """
    if max_em_iters < 1 or tolerance <= 0:
        raise ValueError('Require max_em_iters >= 1 and tolerance > 0.')
    started_all = time.perf_counter()
    arrays = prepared.arrays
    if not 0 < INITIAL_PI_START < 1:
        raise ValueError('INITIAL_PI_START must be strictly between zero and one.')
    gates = (fit_gate_models(arrays, initial_gate_weights(prepared, start_id)) if start_result is None
             else {key: {**fit, 'coef':fit['coef'].copy(), 'table':fit['table'].copy(),
                         'result':copy(fit['result']), 'diagnostics':dict(fit['diagnostics'])}
                   for key,fit in start_result.gate_params.items()})
    pi = start_probability(start_id) if start_result is None else start_result.pi_state2
    beta1 = np.zeros(len(TRANSITION_X_COLS)); beta2 = beta1.copy()
    if start_id and not no_switch:
        persistence = np.random.default_rng([SEED, start_id, 2]).uniform(.4, 2.8)
        beta1[0], beta2[0] = -persistence, persistence
    if start_result is not None and not no_switch:
        beta1, beta2 = start_result.beta_from1.copy(), start_result.beta_from2.copy()

    def e_step(g, p, b1, b2):
        emission = log_emission_by_household_year(arrays, incidence, g)
        if no_switch:
            ll, gamma = no_switch_e_step(arrays, emission, p, arrays_only=True)
            return ll, gamma, None
        return forward_backward(arrays, emission, p, b1, b2, arrays_only=True)

    ll, gamma, xi = e_step(gates, pi, beta1, beta2)
    trans = []
    if not no_switch:
        # Valid coefficient/covariance containers even if the FIRST EM step
        # is rejected. maxiter=0 evaluates the specified zero coefficients.
        trans = list(fit_transition_models(xi, beta1, beta2, arrays, maxiter=0))
    history = []
    stable = 0
    converged = False
    stop_reason = 'max_iterations_reached'
    label = 'No-switch' if no_switch else 'Proposed'
    incidence_ok = all(incidence[c]['result'].success for c in CATEGORY_NAMES)
    for iteration in range(1, max_em_iters + 1):
        started = time.perf_counter()
        first_gamma = gamma[arrays.hh_starts, 1]
        new_pi = float(np.clip(np.average(first_gamma, weights=arrays._hh_weights), PI_EPS, 1-PI_EPS))
        new_trans = [] if no_switch else list(fit_transition_models(xi, beta1, beta2, arrays))
        new_beta1, new_beta2 = (beta1, beta2) if no_switch else tuple(f['coef'] for f in new_trans)
        new_gates = fit_gate_models(arrays, gamma, previous_fits=gates)
        candidate_ll, candidate_gamma, candidate_xi = e_step(new_gates, new_pi, new_beta1, new_beta2)
        candidate_delta = candidate_ll - ll
        failures = sum(not f['result'].success for f in [*new_trans, *new_gates.values()])
        threshold = tolerance + EM_REL_TOL * max(1.0, abs(ll))
        # Tolerance for arithmetic noise, much smaller than an EM stop threshold.
        roundoff = max(1e-8, 64 * np.finfo(float).eps * max(1.0, abs(ll)))
        accepted = bool(np.isfinite(candidate_ll) and candidate_delta >= 0)
        material_drop = not np.isfinite(candidate_ll) or candidate_delta < -roundoff
        delta = 0.0
        if accepted:
            delta = candidate_delta
            pi, gates, trans = new_pi, new_gates, new_trans
            beta1, beta2 = new_beta1, new_beta2
            ll, gamma, xi = candidate_ll, candidate_gamma, candidate_xi
        plateau = not material_drop and delta <= threshold and failures == 0 and incidence_ok
        stable = stable + 1 if plateau else 0
        converged = stable >= EM_STABLE_ITERS
        if material_drop:
            stop_reason = 'downhill_step_rejected'
        elif converged:
            stop_reason = 'stable_likelihood_and_block_gradients'
        history.append({
            'phase': 'EM', 'iteration': iteration, 'log_likelihood': ll, 'pi_state2': pi,
            'log_likelihood_change': delta, 'candidate_log_likelihood_change': candidate_delta,
            'step_accepted': accepted, 'likelihood_parameter_stage': 'after_M_step',
            'iteration_seconds': time.perf_counter()-started,
            'elapsed_seconds': time.perf_counter()-started_all,
            'converged': converged, 'tolerance': tolerance, 'relative_tolerance': EM_REL_TOL,
            'effective_ll_tolerance': threshold, 'stable_iterations': stable,
            'max_em_iters': max_em_iters, 'optimizer_failures': failures,
            'stop_reason': stop_reason,
        })
        if verbose and (iteration == 1 or iteration % EM_PRINT_EVERY == 0 or converged or material_drop or iteration == max_em_iters):
            print(f'{label} EM {iteration:03d}/{max_em_iters}: LL={ll:.6f}, '
                  f'delta={delta:.6g}, candidate_delta={candidate_delta:.6g}, '
                  f'block_failures={failures}, stable={stable}/{EM_STABLE_ITERS}')
        if material_drop:
            warnings.warn(f'{label}: downhill EM proposal rejected; last accepted fit retained.', RuntimeWarning)
            break
        if converged:
            break
    return dict(gates=gates, pi=pi, beta1=beta1, beta2=beta2, trans=trans,
                ll=ll, gamma=posterior_frame(arrays, gamma), history=pd.DataFrame(history), converged=converged,
                stop_reason=stop_reason, estimation_seconds=time.perf_counter()-started_all)

def estimate_model(prepared: PreparedData, max_em_iters: int = MAX_EM_ITERS,
                   tolerance: float = TOL, incidence=None, start_result=None, verbose=True, start_id=0) -> EstimationResult:
    started_all = time.perf_counter()
    if verbose: print('Fitting incidence models; then safeguarded proposed-model EM...')
    if incidence is None: incidence = fit_incidence_models(prepared.arrays)

    switching = bool(len(prepared.arrays.transition_curr_pos))
    out = _run_monotone_em(prepared, incidence, max_em_iters, tolerance,
                           no_switch=not switching, start_result=start_result, verbose=verbose, start_id=start_id)
    for f in [*[incidence[c] for c in CATEGORY_NAMES], *out['trans'], *out['gates'].values()]:
        finalize_diagnostics(f)
    initial_table = pd.DataFrame([{'parameter':'initial_pi_logit','estimate':np.log(out['pi'])-np.log1p(-out['pi'])}])
    transition_fits = dict(zip(('from_state_1', 'from_state_2'), out['trans']))
    tables = [fit['table'] for fit in transition_fits.values()]
    result = EstimationResult(prepared, incidence, out['gates'], out['beta1'], out['beta2'], out['pi'],
        out['gamma'], out['history'], initial_table,
        pd.concat(tables, ignore_index=True) if tables else pd.DataFrame(), out['ll'], transition_fits)
    result.optimizer_diagnostics_df = collect_optimizer_diagnostics(result)
    result.estimation_seconds = time.perf_counter()-started_all
    result.converged = out['converged']
    result.stop_reason = out['stop_reason']
    return result

def collect_optimizer_diagnostics(result: EstimationResult) -> pd.DataFrame:
    """Return one diagnostic row for every final optimized logit block."""
    rows = [result.incidence_models[category]["diagnostics"] for category in CATEGORY_NAMES]
    if result.transition_fits is None:
        raise RuntimeError("Final transition fits are unavailable for optimizer diagnostics.")
    rows.extend(fit["diagnostics"] for fit in result.transition_fits.values())
    rows.extend(result.gate_params[key]["diagnostics"] for key in GATE_KEYS)
    return pd.DataFrame(rows)


In [5]:
@dataclass
class NoSwitchResult:
    """Two latent household classes with exactly zero switching probabilities."""

    prepared: PreparedData
    incidence_models: Dict[str, Any]
    gate_params: Dict[str, Any]
    pi_state2: float
    gamma_df: pd.DataFrame
    log_likelihood: float
    history_df: pd.DataFrame
    converged: bool
    estimation_seconds: float

def no_switch_e_step(arrays, emission_loglik, pi_state2, arrays_only=False):
    """Exact household mixture likelihood for P = [[1, 0], [0, 1]].

    Sum annual log emissions within each household BEFORE mixing the two
    classes. Each household has one latent class for its entire sequence.
    Direct mixture evaluation avoids clipping structural zeros to epsilon.
    """
    if not 0 < pi_state2 < 1:
        raise ValueError("Initial class probability must be strictly between 0 and 1.")
    household_emission = np.add.reduceat(emission_loglik, arrays.hh_starts, axis=0)
    joint = household_emission + np.array([np.log1p(-pi_state2), np.log(pi_state2)])
    household_ll = np.logaddexp(joint[:, 0], joint[:, 1])
    posterior = np.exp(joint - household_ll[:, None])
    posterior /= posterior.sum(axis=1, keepdims=True)
    # Contiguous annual groups use the identical posterior in every year.
    gamma = np.repeat(posterior, arrays.hh_ends - arrays.hh_starts, axis=0)
    total_ll = float(arrays._hh_weights @ household_ll) + arrays._incidence_loglik
    return total_ll, gamma if arrays_only else posterior_frame(arrays, gamma)


def estimate_no_switch_model(proposed, max_em_iters=MAX_EM_ITERS, tolerance=TOL, start_result=None, verbose=True, start_id=0):
    """Re-estimate fixed household classes with the same safeguarded optimizer.

    Reuse the exactly separable incidence fits after raw trip buffers are freed.
    The common mixing probability and gate coefficients are independently estimated.
    """
    out = _run_monotone_em(proposed.prepared, proposed.incidence_models,
                           max_em_iters, tolerance, no_switch=True, start_result=start_result, verbose=verbose, start_id=start_id)
    # Record rank diagnostics; inference is supplied by full bootstrap refits.
    for fit in out['gates'].values():
        finalize_diagnostics(fit)
    result = NoSwitchResult(proposed.prepared, proposed.incidence_models, out['gates'],
        out['pi'], out['gamma'], out['ll'], out['history'], out['converged'], out['estimation_seconds'])
    result.stop_reason = out['stop_reason']
    return result


## 4. Household bootstrap inference and outputs
Both models export every free parameter, bootstrap SE, percentile 95% CI, approximate Wald z/p/stars, and a full bootstrap covariance matrix. `initial_pi_logit` adds one parameter; `initial_probability.csv` reports pi on the probability scale. Proposed posterior profiles are re-estimated in every accepted draw. BIC uses independent households. Fixed nested transitions have no significance tests. Draw-level estimates and diagnostics are checkpointed atomically after every attempted replicate. Aggregate CSVs refresh every `BOOTSTRAP_EXPORT_EVERY` draws and at completion.

For B accepted household bootstrap refits, all reported variances use
$$\widehat{\mathrm{Var}}_{boot}(\hat\theta)=\frac{1}{B-1}\sum_{b=1}^{B}(\hat\theta^{(b)}-\bar\theta_{boot})^2.$$
SEs are the square roots of these variances. Wald z/p/stars use the original-sample estimate and this bootstrap SE. Percentile confidence intervals use the bootstrap draws directly. No block-wise inverse-Hessian variance is used for inference; Hessians remain only for numerical optimization and rank diagnostics.

**Reporting the method:** We estimate the HMM by maximum likelihood using safeguarded EM and estimate a common initial-state probability separately for each model. We obtain inference using a nonparametric household cluster bootstrap: entire household histories are sampled with replacement, all parameters are re-estimated, and state labels are aligned with the original fit. The empirical covariance of accepted bootstrap estimates supplies standard errors; reported Wald tests use these standard errors and confidence intervals use bootstrap percentiles. Report the number of attempted/successful draws and diagnostics alongside the results. This procedure treats households as the independent sampling units.


In [6]:
def build_annual_sufficient_statistics(data):
    return data.annual_stats.copy()

def build_transition_matrix(beta_from1: np.ndarray, beta_from2: np.ndarray, x: np.ndarray) -> pd.DataFrame:
    """Build a two-by-two transition matrix at a supplied annual covariate vector."""
    p12 = float(sigmoid(np.dot(beta_from1, x)))
    p22 = float(sigmoid(np.dot(beta_from2, x)))
    return pd.DataFrame({
        "from_state": ["State 1", "State 2"],
        "to_state_1": [1.0 - p12, 1.0 - p22],
        "to_state_2": [p12, p22],
    })

def build_transition_outputs(result: EstimationResult) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Build baseline and event-specific transition matrices at mean covariates."""
    hy = result.prepared.hy
    mean_x = np.array([
        1.0 if col == "intercept" else float(hy[col].mean())
        for col in TRANSITION_X_COLS
    ], dtype=np.float64)
    baseline = build_transition_matrix(result.beta_from1, result.beta_from2, mean_x)
    rows: List[pd.DataFrame] = []
    for event in TRANSITION_EVENT_COLUMNS:
        position = TRANSITION_X_COLS.index(event)
        for event_value in [0.0, 1.0]:
            x = mean_x.copy()
            x[position] = event_value
            matrix = build_transition_matrix(result.beta_from1, result.beta_from2, x)
            matrix.insert(0, "event", event)
            matrix.insert(1, "event_value", int(event_value))
            rows.append(matrix)
    return baseline, pd.concat(rows, ignore_index=True)

def infer_posterior_ratio_difference(household_stats, numerator_stat, denominator_stat):
    """Point estimates only. Uncertainty is filled from complete bootstrap refits."""
    values = []
    for state in (1, 2):
        denominator = float(household_stats[f'{denominator_stat}_s{state}'].sum())
        numerator = float(household_stats[f'{numerator_stat}_s{state}'].sum())
        values.append(numerator/denominator if denominator > 0 else np.nan)
    difference = values[1]-values[0]
    return dict(state_1_estimate=values[0], state_2_estimate=values[1],
        state_1_standard_error=np.nan, state_2_standard_error=np.nan,
        difference_state2_minus_state1=difference, difference_standard_error=np.nan,
        z_value=np.nan, p_value=np.nan, significance_stars='',
        difference_with_stars=f'{difference:.6f}', inference='bootstrap_pending',
        n_household_clusters=len(household_stats))


def _collapse_profile_statistics_to_households(
    annual: pd.DataFrame,
    gamma_df: pd.DataFrame,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Align the estimation sample and collapse posterior-weighted statistics by household."""
    gamma = gamma_df[[
        "household_code", "panel_year", "gamma_state_1", "gamma_state_2"
    ]].copy()
    if gamma.duplicated(["household_code", "panel_year"]).any():
        raise ValueError("Posterior gamma contains duplicate household-year keys.")

    aligned = annual.merge(
        gamma,
        on=["household_code", "panel_year"],
        how="outer",
        indicator=True,
        validate="one_to_one",
    )
    alignment_summary = pd.DataFrame([{
        "panel_household_years_before_alignment": int(len(annual)),
        "panel_household_years_missing_from_posterior": int((aligned["_merge"] == "left_only").sum()),
        "posterior_household_years_missing_from_panel": int((aligned["_merge"] == "right_only").sum()),
        "matched_household_years_retained": int((aligned["_merge"] == "both").sum()),
    }])
    matched = aligned.loc[aligned["_merge"] == "both"].drop(columns="_merge").reset_index(drop=True)
    if matched.empty:
        raise ValueError("No household-years remain after profile/posterior alignment.")

    excluded = {
        "household_code", "panel_year", "gamma_state_1", "gamma_state_2",
        *TRANSITION_EVENT_COLUMNS,
    }
    stat_columns = [column for column in annual.columns if column not in excluded]
    gamma_state_1 = matched["gamma_state_1"].to_numpy(dtype=np.float64, copy=False)
    gamma_state_2 = matched["gamma_state_2"].to_numpy(dtype=np.float64, copy=False)
    weighted = {"household_code": matched["household_code"]}
    weighted_columns: List[str] = []
    for column in stat_columns:
        values = matched[column].to_numpy(dtype=np.float64, copy=False)
        weighted[f"{column}_s1"] = values * gamma_state_1
        weighted[f"{column}_s2"] = values * gamma_state_2
        weighted_columns.extend([f"{column}_s1", f"{column}_s2"])
    household_stats = (
        pd.DataFrame(weighted)
        .groupby("household_code", sort=False, observed=True)[weighted_columns]
        .sum()
    )
    alignment_summary["matched_households_retained"] = int(len(household_stats))
    return household_stats, alignment_summary

def build_state_profile_summary_with_inference(
    result: EstimationResult,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Build five-category weighted profile point estimates for bootstrap inference."""
    annual = result.prepared.annual_stats
    arrays = result.prepared.arrays
    gamma1,gamma2 = _align_gamma_to_hy(result.gamma_df, arrays)
    weight1,weight2 = gamma1*arrays._hy_weights,gamma2*arrays._hy_weights
    excluded={'household_code','panel_year',*TRANSITION_EVENT_COLUMNS}
    totals={}
    for column in annual.columns:
        if column not in excluded:
            values=annual[column].to_numpy(dtype=float,copy=False)
            totals[column+'_s1']=[float(weight1@values)]
            totals[column+'_s2']=[float(weight2@values)]
    household_stats=pd.DataFrame(totals)
    alignment_summary=pd.DataFrame([dict(panel_household_years_before_alignment=len(annual),
        panel_household_years_missing_from_posterior=0,posterior_household_years_missing_from_panel=0,
        matched_household_years_retained=len(annual),matched_households_retained=len(arrays.hh_starts))])
    incidence_rows: List[Dict[str, Any]] = []
    composition_rows: List[Dict[str, Any]] = []
    wallet_rows: List[Dict[str, Any]] = []

    for category, cfg in CATEGORY_CONFIG.items():
        incidence_inference = infer_posterior_ratio_difference(
            household_stats, f"{category}_purchase_n", "n_trips"
        )
        incidence_rows.append({
            "profile_section": "purchase_incidence",
            "category": category,
            "outcome_code": np.nan,
            "outcome_label": "purchase",
            "state_1_purchase_rate": incidence_inference["state_1_estimate"],
            "state_2_purchase_rate": incidence_inference["state_2_estimate"],
            **{key: value for key, value in incidence_inference.items()
               if key not in {"state_1_estimate", "state_2_estimate"}},
        })

        for outcome_code, outcome_label in cfg["outcomes"].items():
            if outcome_code == 0:
                continue
            composition_inference = infer_posterior_ratio_difference(
                household_stats,
                f"{category}_{outcome_label}_n",
                f"{category}_purchase_n",
            )
            composition_rows.append({
                "profile_section": "conditional_composition",
                "category": category,
                "outcome_code": outcome_code,
                "outcome_label": outcome_label,
                "state_1_conditional_share": composition_inference["state_1_estimate"],
                "state_2_conditional_share": composition_inference["state_2_estimate"],
                **{key: value for key, value in composition_inference.items()
                   if key not in {"state_1_estimate", "state_2_estimate"}},
            })

        wallet_inference = infer_posterior_ratio_difference(
            household_stats, f"{category}_spend_sum", "food_total_spend_sum"
        )
        wallet_rows.append({
            "profile_section": "wallet_share",
            "category": category,
            "outcome_code": np.nan,
            "outcome_label": "overall_category_wallet_share",
            "state_1_food_wallet_share": wallet_inference["state_1_estimate"],
            "state_2_food_wallet_share": wallet_inference["state_2_estimate"],
            **{key: value for key, value in wallet_inference.items()
               if key not in {"state_1_estimate", "state_2_estimate"}},
        })

    incidence = pd.DataFrame(incidence_rows)
    composition = pd.DataFrame(composition_rows)
    wallet = pd.DataFrame(wallet_rows)
    summary = pd.concat([incidence, composition, wallet], ignore_index=True, sort=False)
    preferred_columns = [
        "profile_section", "category", "outcome_code", "outcome_label",
        "state_1_purchase_rate", "state_2_purchase_rate",
        "state_1_conditional_share", "state_2_conditional_share",
        "state_1_food_wallet_share", "state_2_food_wallet_share",
        "state_1_standard_error", "state_2_standard_error",
        "difference_state2_minus_state1", "difference_standard_error",
        "z_value", "p_value", "significance_stars", "difference_with_stars",
        "inference", "n_household_clusters",
    ]
    summary = summary.reindex(columns=preferred_columns)
    for table in [summary,incidence,composition,wallet]:
        table['n_household_clusters']=int(arrays._hh_weights.sum())
    valid_stars = {"", "*", "**", "***"}
    observed_stars = set(summary["significance_stars"].fillna("").astype(str).unique())
    if not observed_stars.issubset(valid_stars):
        raise AssertionError(f"Unexpected significance-star values: {sorted(observed_stars - valid_stars)}")
    if len(incidence) != len(CATEGORY_CONFIG) or len(wallet) != len(CATEGORY_CONFIG):
        raise AssertionError("State-profile output does not contain all five categories.")
    expected_composition = sum(len(cfg["outcomes"]) - 1 for cfg in CATEGORY_CONFIG.values())
    if len(composition) != expected_composition:
        raise AssertionError("State-profile output does not contain every positive outcome.")
    return summary, incidence, composition, wallet, alignment_summary


In [7]:
def fitted_blocks(result, proposed):
    blocks=[(f'{c}_incidence',result.incidence_models[c],False) for c in CATEGORY_NAMES]
    if proposed:
        blocks += [(f'transition_{name}',fit,True) for name,fit in result.transition_fits.items()]
    blocks += [(f'choice_{k}',result.gate_params[k],True) for k in GATE_KEYS]
    return blocks


def parameter_table(result, proposed):
    frames=[]
    for name, fit, latent in fitted_blocks(result, proposed):
        frames.append(pd.DataFrame({'equation_name':name, 'parameter':fit['table'].parameter,
                                    'estimate':fit['coef']}))
    frames.append(pd.DataFrame([dict(equation_name='initial_state', parameter='initial_pi_logit',
        estimate=np.log(result.pi_state2)-np.log1p(-result.pi_state2))]))
    table=pd.concat(frames, ignore_index=True)
    assert len(table)==(47 + (len(result.transition_fits)*len(TRANSITION_X_COLS) if proposed else 0)) and not table.parameter.duplicated().any()
    return table


def coefficient_outputs(result, proposed):
    table=parameter_table(result, proposed)
    draws=getattr(result, 'bootstrap_coefficients', np.empty((0,len(table))))
    valid=getattr(result, 'bootstrap_inference_valid', False)
    table['bootstrap_variance']=np.var(draws, axis=0, ddof=1) if valid else np.nan
    table['standard_error']=np.sqrt(table.bootstrap_variance)
    table['z_statistic']=table.estimate/table.standard_error.where(table.standard_error>0)
    table['p_value']=2*norm.sf(abs(table.z_statistic))
    table['significance_stars']=table.p_value.map(significance_stars)
    table['ci_95_lower']=np.quantile(draws, .025, axis=0) if valid else np.nan
    table['ci_95_upper']=np.quantile(draws, .975, axis=0) if valid else np.nan
    table['estimate_with_stars']=[f'{v:.6f}{star}' for v,star in zip(table.estimate,table.significance_stars)]
    table['covariance_method']='household cluster bootstrap; all model parameters refitted'
    table['ci_method']='percentile bootstrap'
    table['p_value_method']='normal-reference Wald using bootstrap SE; initial_pi_logit null = 0 (pi = 0.5)'
    table['bootstrap_successful_draws']=len(draws)
    table['inference_status']='bootstrap' if valid else 'unavailable: inspect bootstrap diagnostics'
    cov=np.cov(draws,rowvar=False,ddof=1) if valid else np.full((len(table),len(table)),np.nan)
    return table,pd.DataFrame(cov,index=table.parameter,columns=table.parameter)


def save_outputs(proposed,nested,destination):
    """Organized machine-readable and reviewable outputs for both fitted models."""
    root=Path(destination); root.mkdir(parents=True,exist_ok=True)
    comparison=[]
    for name,result,is_proposed in [('proposed_hmm',proposed,True),('nested_no_switch',nested,False)]:
        folder=root/('02_'+name if is_proposed else '03_'+name); folder.mkdir(exist_ok=True)
        coef,cov=coefficient_outputs(result,is_proposed)
        coef.to_csv(folder/'01_full_coefficient_table.csv',index=False)
        cov.to_csv(folder/'02_bootstrap_covariance.csv',index_label='parameter')
        result.gamma_df.rename(columns={'household_code':'hhid'}).to_csv(folder/'06_posterior_household_year.csv',index=False)
        result.history_df.to_csv(folder/'07_estimation_log.csv',index=False)
        diagnostics=pd.DataFrame([f['diagnostics'] for _,f,_ in fitted_blocks(result,is_proposed)])
        diagnostics.to_csv(folder/'09_optimizer_diagnostics.csv',index=False)
        n=len(result.prepared.arrays.hh_starts); ll=result.log_likelihood; k=len(coef)
        row=dict(model=name,log_likelihood=ll,n_parameters=k,AIC=-2*ll+2*k,BIC=-2*ll+k*np.log(n),
            n_households=n,n_household_years=len(result.prepared.hy),n_trips=len(result.prepared.df),
            estimated_initial_pi=result.pi_state2,converged=result.converged,stop_reason=result.stop_reason,
            em_iterations=int(result.history_df.iteration.max()),estimation_seconds=result.estimation_seconds,
            BIC_sample_size_definition='independent households',
            rank_deficient_blocks=int((diagnostics.hessian_rank<diagnostics.parameter_count).sum()))
        row['observed_transitions'] = len(result.prepared.arrays.transition_curr_pos)
        row['bootstrap_successful_draws']=len(getattr(result, 'bootstrap_coefficients', []))
        row['bootstrap_inference_valid']=getattr(result, 'bootstrap_inference_valid', False)
        pd.DataFrame([initial_probability_summary(result)]).to_csv(folder/'initial_probability.csv',index=False)
        comparison.append(row)
        pd.DataFrame([row]).to_csv(folder/'05_model_fit.csv',index=False)
        if is_proposed:
            if result.transition_fits:
                baseline,events=build_transition_outputs(result)
                baseline['event']='baseline_at_mean_events'
                pd.concat([baseline,events],ignore_index=True).to_csv(folder/'03_transition_matrices_by_event.csv',index=False)
            else:
                pd.DataFrame(columns=['from_state','to_state_1','to_state_2','event','event_value']).to_csv(
                    folder/'03_transition_matrices_by_event.csv',index=False)
            summary=bootstrap_profile_summary(result)
            summary['signed_difference_with_stars']=[f'{v:+.6f}{s}' if np.isfinite(v) else '' for v,s in zip(summary.difference_state2_minus_state1,summary.significance_stars)]
            summary.to_csv(folder/'04_posterior_purchase_profiles.csv',index=False)
            healthy=summary.loc[(summary.profile_section=='conditional_composition') &
                                (summary.outcome_code==1)].copy()
            healthy.to_csv(folder/'11_posterior_healthy_choices.csv',index=False)
            print('\nPosterior conditional healthy-choice probabilities (not spending shares):')
            print(healthy[['category','state_1_conditional_share','state_2_conditional_share',
                           'signed_difference_with_stars']].to_string(index=False))
            fv=summary.loc[(summary.category=='fv')&(summary.outcome_label=='fresh')].iloc[0]
            label_ok=bool(fv.difference_state2_minus_state1>0)
            pd.DataFrame([dict(state_2_has_higher_fresh_share=label_ok,
                interpretation='Review all five category profiles before labeling states Healthy/Unhealthy')]).to_csv(folder/'10_state_label_diagnostic.csv',index=False)
            if not label_ok: warnings.warn('State 2 has no higher fresh-FV profile. Inspect initialization/local optimum before using Healthy label.')
        else:
            pd.DataFrame(np.eye(2),index=['from_state_1','from_state_2'],columns=['to_state_1','to_state_2']).to_csv(folder/'03_fixed_transition_matrix.csv')
    comp=pd.DataFrame(comparison)
    for m in ['AIC','BIC']: comp['delta_'+m]=comp[m]-comp[m].min()
    provisional=(not comp.converged.all()) or proposed.log_likelihood<nested.log_likelihood-TOL or comp.rank_deficient_blocks.gt(0).any()
    comp['comparison_status']='provisional: inspect convergence, likelihood and Hessians' if provisional else 'EM criteria met; local optima remain possible'
    if not len(proposed.prepared.arrays.transition_curr_pos):
        comp['comparison_status']='All histories are singletons. Both specifications reduce to one fixed-class mixture.'
    folder=root/'01_model_comparison'; folder.mkdir(exist_ok=True)
    comp.to_csv(folder/'nested_vs_proposed.csv',index=False)
    pd.DataFrame([proposed.prepared.df.attrs['sorting_audit']]).to_csv(root/'sample_audit.csv',index=False)
    config=dict(version='V42',source_version='V39 choice HMM',
        categories=CATEGORY_NAMES,
        beverage_rule='milk_y in (1,2) OR original beverage_y=1 => 1; otherwise original beverage_y',
        milk_scope='existing ordinary-milk category; no sweetened/flavored milk changes',
        yogurt_definition='panel yogurt_y: 1 regular sugar, 2 high sugar',
        year_min=YEAR_MIN,year_max=YEAR_MAX,full_sample_starts=N_STARTS,
        input_path=proposed.prepared._source_path,run_mode=RUN_MODE,transition_covariates=TRANSITION_EVENT_COLUMNS,
        choice_covariates=GATE_COVARIATE_COLS,initial_pi='estimated separately in each model',minimum_years=MIN_OBSERVED_YEARS_PER_HOUSEHOLD,
        bootstrap_reps=BOOTSTRAP_REPS if RUN_MODE!='synthetic' else SYNTHETIC_BOOTSTRAP_REPS,bootstrap_seed=BOOTSTRAP_SEED,
        bootstrap_min_success=BOOTSTRAP_MIN_SUCCESS,bootstrap_min_success_fraction=BOOTSTRAP_MIN_SUCCESS_FRACTION,
        bootstrap_max_em_iters=BOOTSTRAP_MAX_EM_ITERS,bootstrap_boundary_tol=BOOTSTRAP_BOUNDARY_TOL,
        bootstrap_workers=str(BOOTSTRAP_WORKERS),bootstrap_resume=BOOTSTRAP_RESUME,
        bootstrap_export_every=BOOTSTRAP_EXPORT_EVERY,incidence_cache=proposed.prepared.df.attrs['incidence_cache'],
        implementation='V42_fast_bootstrap_v2',compiled_kernels=bool(USE_NUMBA and numba is not None),incidence_likelihood_factorized=True,
        gap_policy=YEAR_GAP_POLICY,max_em_iters=MAX_EM_ITERS,absolute_tolerance=TOL,relative_tolerance=EM_REL_TOL,
        numpy_version=np.__version__,pandas_version=pd.__version__,duckdb_version=duckdb.__version__)
    (root/'run_configuration.json').write_text(json.dumps(config,indent=2),encoding='utf-8')
    (root/'README.md').write_text(
        'V42 sugar-panel purchase-choice estimates. See sample_audit.csv for the retained sample.\n'
        'multistart_diagnostics.csv identifies the selected full-sample starts for both models.\n'
        'yogurt_y=1 is regular sugar and yogurt_y=2 is high sugar. Any milk gives combined beverage code 1.\n'
        'State 2 is oriented by the fresh-FV gate. Profiles are posterior-weighted sample statistics.\n'
        'Variance and percentile intervals use household bootstrap refits. See 04_bootstrap for success counts.\n'
        'Wald stars: * p<.10, ** p<.05, *** p<.01. BIC counts independent household histories.\n'
        'A single observed year per household identifies a mixture only. See comparison_status.\n',
        encoding='utf-8')
    print(comp[['model','log_likelihood','n_parameters','AIC','BIC','converged']].to_string(index=False))
    return comp


In [8]:
# Household cluster bootstrap. One integer weight per complete household history.
from copy import copy
from types import SimpleNamespace


def swap_state_labels(result, proposed):
    """Exact likelihood-preserving permutation for state_num coded as 1 and 2."""
    for fit in result.gate_params.values():
        beta=fit['coef'].copy()
        beta[0],beta[1]=beta[0]+3*beta[1],-beta[1]
        fit['coef']=beta
        fit['table']['estimate']=beta
        fit['result'].x=beta.copy()
    if proposed and result.transition_fits:
        beta1,beta2=-result.beta_from2.copy(),-result.beta_from1.copy()
        result.beta_from1,result.beta_from2=beta1,beta2
        for key,beta in [('from_state_1',beta1),('from_state_2',beta2)]:
            fit=result.transition_fits[key]
            fit['coef']=beta
            fit['table']['estimate']=beta
            fit['result'].x=beta.copy()
        result.transition_table=pd.concat([f['table'] for f in result.transition_fits.values()],ignore_index=True)
    result.pi_state2=1-result.pi_state2
    gamma=result.gamma_df
    gamma[['gamma_state_1','gamma_state_2']]=gamma[['gamma_state_2','gamma_state_1']].to_numpy()
    gamma['decoded_state']=3-gamma.decoded_state
    result.history_df['pi_state2']=1-result.history_df.pi_state2
    if proposed:
        result.initial_state_table=pd.DataFrame([{'parameter':'initial_pi_logit',
            'estimate':np.log(result.pi_state2)-np.log1p(-result.pi_state2)}])


def canonicalize_state_labels(result, proposed):
    # Positive fresh-FV state coefficient defines State 2, with all other
    # categories free to have either direction. This does not constrain the fit.
    if result.gate_params['fv_fresh']['coef'][1] < 0:
        swap_state_labels(result, proposed)


def gate_signature(result, reference):
    """Compare labels on the same fixed covariate grid across bootstrap draws."""
    profiles=[]
    for key in GATE_KEYS:
        base=reference.prepared.arrays.gate_blocks[key].x_base
        base=base[np.linspace(0,len(base)-1,min(128,len(base)),dtype=int)]
        beta=result.gate_params[key]['coef']
        shared=base @ np.r_[beta[0],beta[2:]]
        profiles.append(expit(shared[:,None]+np.array([1.,2.])*beta[1]))
    return np.vstack(profiles)


def align_bootstrap_labels(result, reference, proposed):
    target=gate_signature(reference,reference)
    value=gate_signature(result,reference)
    direct=float(np.mean((value-target)**2))
    swapped=float(np.mean((value[:,::-1]-target)**2))
    if swapped < direct:
        swap_state_labels(result,proposed)
    return dict(labels_swapped=swapped<direct,label_match_distance=min(direct,swapped),
                label_match_margin=abs(direct-swapped))


def fit_acceptance(result, proposed):
    fits=[f for _,f,_ in fitted_blocks(result,proposed)]
    issues=[]
    if not result.converged: issues.append('EM_nonconvergence:'+result.stop_reason)
    if not result.history_df.step_accepted.any(): issues.append('no_accepted_EM_updates')
    if not np.isfinite(result.log_likelihood): issues.append('nonfinite_likelihood')
    if not np.isfinite(parameter_table(result,proposed).estimate).all(): issues.append('nonfinite_parameters')
    if any(not f['result'].success for f in fits): issues.append('block_optimizer_failure')
    if any(f['diagnostics']['hessian_rank']<len(f['coef']) for f in fits): issues.append('rank_deficient_block')
    if min(result.pi_state2,1-result.pi_state2)<=BOOTSTRAP_BOUNDARY_TOL: issues.append('pi_at_boundary')
    signature=gate_signature(result,result)
    if np.max(np.abs(signature[:,1]-signature[:,0]))<1e-6: issues.append('collapsed_states')
    return issues


def profile_values(result):
    summary=build_state_profile_summary_with_inference(result)[0]
    state1=summary[['state_1_purchase_rate','state_1_conditional_share','state_1_food_wallet_share']].bfill(axis=1).iloc[:,0]
    state2=summary[['state_2_purchase_rate','state_2_conditional_share','state_2_food_wallet_share']].bfill(axis=1).iloc[:,0]
    return summary, np.column_stack([state1,state2,state2-state1])


def initial_probability_summary(result):
    draws=getattr(result,'bootstrap_coefficients',np.empty((0,1)))
    valid=getattr(result,'bootstrap_inference_valid',False)
    probabilities=expit(draws[:,-1])
    variance=float(np.var(probabilities,ddof=1)) if valid else np.nan
    return dict(parameter='pi_state2',estimate=result.pi_state2,
        bootstrap_variance=variance,standard_error=float(np.sqrt(variance)),
        ci_95_lower=float(np.quantile(probabilities,.025)) if valid else np.nan,
        ci_95_upper=float(np.quantile(probabilities,.975)) if valid else np.nan,
        ci_method='percentile household bootstrap',bootstrap_successful_draws=len(draws),
        inference_status='bootstrap' if valid else 'unavailable: inspect bootstrap diagnostics')


def bootstrap_profile_summary(result):
    summary,points=profile_values(result)
    draws=getattr(result,'bootstrap_profiles',np.empty((0,len(summary),3)))
    valid=getattr(result,'bootstrap_inference_valid',False)
    if valid:
        variance=np.var(draws,axis=0,ddof=1)
        se=np.sqrt(variance)
        low,high=np.quantile(draws,[.025,.975],axis=0)
    else:
        variance=se=low=high=np.full(points.shape,np.nan)
    for j,name in enumerate(['state_1','state_2','difference']):
        summary[name+'_bootstrap_variance']=variance[:,j]
        summary[name+'_standard_error']=se[:,j]
        summary[name+'_ci_95_lower']=low[:,j]
        summary[name+'_ci_95_upper']=high[:,j]
    summary['z_value']=summary.difference_state2_minus_state1/summary.difference_standard_error.where(summary.difference_standard_error>0)
    summary['p_value']=2*norm.sf(abs(summary.z_value))
    summary['significance_stars']=summary.p_value.map(significance_stars)
    summary['difference_with_stars']=[f'{v:.6f}{star}' for v,star in zip(summary.difference_state2_minus_state1,summary.significance_stars)]
    summary['inference']='bootstrap' if valid else 'unavailable: inspect bootstrap diagnostics'
    summary['inference_method']='household cluster bootstrap; refitted posterior weights; percentile CI; normal-reference Wald p'
    summary['bootstrap_successful_draws']=len(draws)
    return summary


# Atomic, deterministic checkpoints; optional shared-memory parallel draws.
import hashlib
import types
from concurrent.futures import ThreadPoolExecutor, wait, FIRST_COMPLETED
from threadpoolctl import threadpool_limits


def _code_fingerprint(function):
    """Hash executable code without notebook cell IDs or source-line numbers."""
    def describe(value):
        if isinstance(value,types.CodeType):
            return (value.co_code.hex(),value.co_names,value.co_varnames,value.co_argcount,
                    value.co_kwonlyargcount,tuple(describe(v) for v in value.co_consts))
        if isinstance(value,tuple): return tuple(describe(v) for v in value)
        if isinstance(value,frozenset): return tuple(sorted(describe(v) for v in value))
        return repr(value)
    return hashlib.sha256(repr(describe(function.__code__)).encode()).hexdigest()


def bootstrap_manifest(proposed,nested):
    prepared=proposed.prepared
    if not hasattr(prepared,'_source_sha256'):
        digest=hashlib.sha256()
        with open(prepared._source_path,'rb') as handle:
            for block in iter(lambda:handle.read(8*1024**2),b''):digest.update(block)
        prepared._source_sha256=digest.hexdigest()
    functions=[_run_monotone_em,_minimize_logit,fit_gate_count_logit,fit_binomial_count_logit,
        fit_gate_models,fit_transition_models,fit_incidence_models,prepare_incidence_cache,
        log_emission_by_household_year,forward_backward,no_switch_e_step,set_household_weights,
        fit_acceptance,profile_values,build_state_profile_summary_with_inference,
        swap_state_labels,align_bootstrap_labels,gate_signature,
        _finish_logit_fit,_optimizer_diagnostics,_cached_value_gradient,finalize_diagnostics,
        estimate_model,estimate_no_switch_model,_align_gamma_to_hy,parameter_table,
        initial_gate_weights,sigmoid,safe_log,infer_posterior_ratio_difference,
        prepare_parquet,_prepare_parquet,start_probability,full_sample_multistart,register_combined_beverage_view,run_household_bootstrap]
    config=dict(implementation='V42_sugar_choice', year_min=YEAR_MIN, year_max=YEAR_MAX, full_sample_starts=N_STARTS, start_seed=SEED,input_sha256=prepared._source_sha256,
        n_households=len(prepared.arrays.hh_starts),n_household_years=len(prepared.hy),
        n_trips=len(prepared.df),seed=BOOTSTRAP_SEED,categories=CATEGORY_CONFIG,
        gate_covariates=GATE_COVARIATE_COLS,transition_covariates=TRANSITION_EVENT_COLUMNS,
        min_years=MIN_OBSERVED_YEARS_PER_HOUSEHOLD,gap_policy=YEAR_GAP_POLICY,
        tolerance=TOL,relative_tolerance=EM_REL_TOL,stable_iterations=EM_STABLE_ITERS,
        opt_ftol=OPT_FTOL,opt_gtol=OPT_GTOL,opt_newton_steps=OPT_NEWTON_STEPS,
        opt_fast_newton_steps=OPT_FAST_NEWTON_STEPS,bootstrap_max_iters=BOOTSTRAP_MAX_EM_ITERS,
        pi_eps=PI_EPS,pi_boundary_tol=BOOTSTRAP_BOUNDARY_TOL,
        numpy_version=np.__version__,pandas_version=pd.__version__,duckdb_version=duckdb.__version__,
        compiled_kernels=bool(USE_NUMBA and numba is not None),
        numba_version=numba.__version__ if USE_NUMBA and numba is not None else None,
        numerical_code={f.__name__:_code_fingerprint(getattr(f, 'py_func', f))
                        for f in [*functions, posterior_frame, compact_bootstrap_histories, _chain_posteriors, _gate_counts, _incidence_counts]})
    return dict(config=config,
        original_fit_issues={'proposed_hmm':fit_acceptance(proposed,True),'nested_no_switch':fit_acceptance(nested,False)},
        original_parameters={name:parameter_table(r,flag).estimate.tolist()
            for name,r,flag in [('proposed_hmm',proposed,True),('nested_no_switch',nested,False)]},
        original_likelihoods=[proposed.log_likelihood,nested.log_likelihood])


def _atomic_json(path,data):
    temporary=path.with_suffix(path.suffix+'.tmp')
    temporary.write_text(json.dumps(data,indent=2),encoding='utf-8')
    os.replace(temporary,path)


def compact_bootstrap_histories(prepared):
    """Work only on sampled histories after incidence has used the full cache index."""
    arrays = prepared.arrays
    keep_hh = arrays._hh_weights > 0
    if keep_hh.all():
        return
    keep_year = arrays._hy_weights > 0
    selected = np.flatnonzero(keep_year)
    # Alternative initialization must still standardize over the original full sample.
    prepared._initialization_panel = prepared.hy
    prepared._initialization_rows = selected
    prepared.hy = prepared.hy.iloc[selected].reset_index(drop=True)
    prepared.annual_stats = prepared.annual_stats.iloc[selected].reset_index(drop=True)
    arrays.household_year = prepared.hy
    arrays.hy_key = pd.MultiIndex.from_frame(prepared.hy[['household_code', 'panel_year']])
    arrays._annual_statistics = prepared.annual_stats
    lengths = (arrays.hh_ends - arrays.hh_starts)[keep_hh]
    arrays.hh_ends = np.cumsum(lengths)
    arrays.hh_starts = arrays.hh_ends - lengths
    arrays.households = prepared.hy.household_code.to_numpy()[arrays.hh_starts].tolist()
    arrays._hh_weights = arrays._hh_weights[keep_hh]
    arrays._hy_weights = arrays._hy_weights[selected]
    arrays.transition_x = arrays.transition_x[selected]
    design, pattern = arrays._annual_event_patterns
    arrays._annual_event_patterns = design, pattern[selected]
    has_previous = np.ones(len(selected), dtype=bool)
    has_previous[arrays.hh_starts] = False
    arrays.transition_curr_pos = np.flatnonzero(has_previous)
    arrays.transition_row_of_hy = np.full(len(selected), -1, dtype=np.int64)
    arrays.transition_row_of_hy[has_previous] = np.arange(has_previous.sum())
    for name in ('_transition_grouping', '_sequence_batches'):
        if hasattr(arrays, name):
            delattr(arrays, name)

    new_position = np.full(len(keep_year), -1, dtype=np.int64)
    new_position[selected] = np.arange(len(selected))
    blocks = {}
    for key, source in arrays.gate_blocks.items():
        block = copy(source)
        active = keep_year[source.hy_pos]
        block.hy_pos = new_position[source.hy_pos[active]]
        block.design_pos = source.design_pos[active]
        block.target = source.target[active]
        block.success_count = source.success_count[active]
        block.failure_count = source.failure_count[active]
        blocks[key] = block
    arrays.gate_blocks = blocks


def run_household_bootstrap(proposed,nested,destination,reps=BOOTSTRAP_REPS,
                            min_success=BOOTSTRAP_MIN_SUCCESS,min_fraction=BOOTSTRAP_MIN_SUCCESS_FRACTION,
                            workers=None):
    """Exact household bootstrap, full refits, reproducible order and safe resume.

    BOOTSTRAP_WORKERS='auto' times two serial draws and two concurrent draws,
    then tries four when two improve throughput. All calibration draws count toward B.
    Workers share read-only caches, own their weights/fits, and use private
    DuckDB connections in disk mode. BLAS threads are capped to prevent nesting.
    No draw is replaced merely because its numerical fit failed.
    """
    if not isinstance(reps,int) or reps<2:raise ValueError('BOOTSTRAP_REPS must be an integer >= 2.')
    if not 0<=min_fraction<=1 or min_success<2:raise ValueError('Invalid bootstrap acceptance threshold.')
    workers=BOOTSTRAP_WORKERS if workers is None else workers
    if workers!='auto' and (not isinstance(workers,int) or workers<1):raise ValueError('Workers must be auto or a positive integer.')
    folder=Path(destination)/'04_bootstrap';folder.mkdir(parents=True,exist_ok=True)
    checkpoint_dir=folder/('checkpoints' if BOOTSTRAP_RESUME else f'checkpoints_{time.time_ns()}')
    checkpoint_dir.mkdir(exist_ok=True)
    manifest=bootstrap_manifest(proposed,nested)
    manifest_path=checkpoint_dir/'manifest.json'
    def compatible(path):
        old=json.loads(path.read_text(encoding='utf-8'))
        if old['config'] != json.loads(json.dumps(manifest['config'])):
            return False
        if old['original_fit_issues'] != manifest['original_fit_issues']:
            return False
        for name, values in manifest['original_parameters'].items():
            if not np.allclose(old['original_parameters'][name],values,rtol=1e-8,atol=1e-6):
                return False
        return np.allclose(old['original_likelihoods'],manifest['original_likelihoods'],rtol=0,atol=1e-4)

    if manifest_path.exists() and not compatible(manifest_path):
        # A replaced parquet or updated model may share the same visible year settings.
        # Keep its draws separate, and resume a compatible alternative when available.
        for candidate in sorted(folder.glob('checkpoints_run_*/manifest.json')):
            if compatible(candidate):
                checkpoint_dir, manifest_path = candidate.parent, candidate
                break
        else:
            checkpoint_dir=folder/f'checkpoints_run_{time.time_ns()}'
            checkpoint_dir.mkdir()
            manifest_path=checkpoint_dir/'manifest.json'
        print(f'Using sample-compatible checkpoint directory: {checkpoint_dir.name}')
    if not manifest_path.exists():
        _atomic_json(manifest_path,manifest)
    references={'proposed_hmm':(proposed,True),'nested_no_switch':(nested,False)}
    names={name:parameter_table(ref,flag).parameter.tolist() for name,(ref,flag) in references.items()}
    base_issues={name:fit_acceptance(ref,flag) for name,(ref,flag) in references.items()}
    for name,issues in base_issues.items():
        if issues:warnings.warn(f'{name}: original fit unsuitable for bootstrap inference: {issues}')
    metadata,_=profile_values(proposed)
    metadata=metadata[['profile_section','category','outcome_code','outcome_label']].copy()
    H=len(proposed.prepared.arrays.hh_starts)
    seeds={b:int(seed.generate_state(1,dtype=np.uint64)[0])
           for b,seed in enumerate(np.random.SeedSequence(BOOTSTRAP_SEED).spawn(reps),1)}
    required=max(min_success,int(math.ceil(min_fraction*reps)))
    if required>reps:warnings.warn(f'Need {required} successful draws; only {reps} requested. Inference will be withheld.')
    completed={}
    for b in range(1,reps+1):
        path=checkpoint_dir/f'replicate_{b:06d}.npz'
        if not path.exists():continue
        try:
            with np.load(path,allow_pickle=False) as saved:
                records=json.loads(str(saved['records'].item()))
                if len(records)!=2 or any(r['replicate']!=b or int(r['draw_seed'])!=seeds[b] for r in records):
                    raise ValueError('Invalid checkpoint seed/replicate.')
                payload={'records':records}
                for record in records:
                    name=record['model']
                    if record['accepted']:
                        values=saved[name].copy()
                        if values.shape!=(len(names[name]),) or not np.isfinite(values).all():raise ValueError('Invalid coefficient checkpoint.')
                        payload[name]=values
                        if name=='proposed_hmm':
                            profiles=saved['profiles'].copy()
                            if profiles.shape!=(len(metadata),3) or not np.isfinite(profiles).all():raise ValueError('Invalid profile checkpoint.')
                            payload['profiles']=profiles
                completed[b]=payload
        except Exception as exc:
            quarantine=path.with_suffix(f'.corrupt_{time.time_ns()}')
            path.rename(quarantine)
            warnings.warn(f'Checkpoint {b} could not be verified ({exc}); retained as {quarantine.name}; rerunning that same draw.')
    if completed:print(f'Resuming: {len(completed)}/{reps} completed draws loaded; no refitting of those draws.',flush=True)

    def draw_one(b):
        started=time.perf_counter()
        rng=np.random.default_rng(seeds[b]);multiplicity=np.bincount(rng.integers(0,H,size=H),minlength=H)
        prepared=copy(proposed.prepared);prepared.arrays=copy(proposed.prepared.arrays)
        set_household_weights(prepared,multiplicity)
        prepared.arrays._incidence_starts={c:proposed.incidence_models[c]['coef'] for c in CATEGORY_NAMES}
        shared_error='';incidence=None;payload={};records=[]
        if not all(base_issues.values()):
            try:
                incidence=fit_incidence_models(prepared.arrays)
                for fit in incidence.values():finalize_diagnostics(fit)
                compact_bootstrap_histories(prepared)
            except Exception as exc:shared_error=f'{type(exc).__name__}: {exc}'
        for name,(reference,flag) in references.items():
            record=dict(replicate=b,draw_seed=str(seeds[b]),model=name,
                unique_sampled_households=int((multiplicity>0).sum()),sampled_households=int(multiplicity.sum()),
                accepted=False,labels_swapped=False,log_likelihood=np.nan,pi_state2=np.nan,
                converged=False,error='',status='failed')
            try:
                if base_issues[name]:raise RuntimeError('Original fit invalid: '+'; '.join(base_issues[name]))
                if shared_error:raise RuntimeError('Incidence refit failed: '+shared_error)
                # Retry an unsuccessful numerical fit on the identical resample.
                for attempt in range(2):
                    kwargs=dict(max_em_iters=BOOTSTRAP_MAX_EM_ITERS, verbose=False,
                                start_result=reference if attempt==0 else None, start_id=0 if attempt==0 else 1)
                    record['fit_attempts']=attempt+1
                    try:
                        if flag:
                            result=estimate_model(prepared, incidence=incidence, **kwargs)
                        else:
                            result=estimate_no_switch_model(SimpleNamespace(prepared=prepared,incidence_models=incidence), **kwargs)
                    except (ValueError, FloatingPointError, np.linalg.LinAlgError) as exc:
                        if attempt: raise
                        record['first_attempt_issue']=str(exc)
                        continue
                    issues=fit_acceptance(result,flag)
                    if not issues: break
                    if not attempt: record['first_attempt_issue']='; '.join(issues)
                record.update(align_bootstrap_labels(result,reference,flag))
                record.update(log_likelihood=result.log_likelihood,pi_state2=result.pi_state2,
                    converged=result.converged,em_iterations=len(result.history_df))
                issues=fit_acceptance(result,flag)
                if record['label_match_margin']<1e-10:issues.append('ambiguous_state_matching')
                table=parameter_table(result,flag);assert table.parameter.tolist()==names[name]
                if flag:
                    _,values=profile_values(result)
                    if not np.isfinite(values).all():issues.append('nonfinite_profile')
                if issues:record.update(status='excluded',error='; '.join(issues))
                else:
                    record.update(accepted=True,status='accepted');payload[name]=table.estimate.to_numpy(copy=True)
                    if flag:payload['profiles']=values
            except Exception as exc:record['error']=f'{type(exc).__name__}: {exc}'
            record['elapsed_seconds']=time.perf_counter()-started;records.append(record)
        payload['records']=records
        target=checkpoint_dir/f'replicate_{b:06d}.npz';temporary=target.with_suffix('.npz.tmp')
        with open(temporary,'wb') as handle:
            np.savez_compressed(handle,records=np.array(json.dumps(records)),
                **{k:v for k,v in payload.items() if k!='records'})
        os.replace(temporary,target)
        return b,payload

    def export_aggregates():
        records=[record for b in sorted(completed) for record in completed[b]['records']]
        pd.DataFrame(records).to_csv(folder/'replicate_diagnostics.csv',index=False)
        for name in references:
            ids=[b for b in sorted(completed) if name in completed[b]]
            table=pd.DataFrame([completed[b][name] for b in ids],columns=names[name])
            table.insert(0,'replicate',ids);table.to_csv(folder/(name+'_coefficient_draws.csv'),index=False)
        frames=[]
        for b in sorted(completed):
            if 'profiles' in completed[b]:
                frame=metadata.copy();frame.insert(0,'replicate',b)
                frame[['state_1_estimate','state_2_estimate','difference']]=completed[b]['profiles'];frames.append(frame)
        table=pd.concat(frames,ignore_index=True) if frames else pd.DataFrame(columns=['replicate',*metadata.columns,'state_1_estimate','state_2_estimate','difference'])
        table.to_csv(folder/'proposed_profile_draws.csv',index=False)

    def accept(answer):
        b,payload=answer;completed[b]=payload
        successes=', '.join(f'{name}: {sum(name in v for v in completed.values())}' for name in references)
        print(f'Bootstrap {b} saved ({len(completed)}/{reps} complete); accepted {successes}',flush=True)
        if len(completed)%BOOTSTRAP_EXPORT_EVERY==0:export_aggregates()

    def run_batch(indices,count):
        if count==1:
            for b in indices:accept(draw_one(b))
            return
        executor=ThreadPoolExecutor(max_workers=count)
        iterator=iter(indices);pending={}
        try:
            for _ in range(count):
                b=next(iterator,None)
                if b is not None:pending[executor.submit(draw_one,b)]=b
            while pending:
                done,_=wait(pending,return_when=FIRST_COMPLETED)
                for future in done:
                    pending.pop(future);accept(future.result())
                    b=next(iterator,None)
                    if b is not None:pending[executor.submit(draw_one,b)]=b
        finally:
            executor.shutdown(wait=True,cancel_futures=True)

    remaining=[b for b in range(1,reps+1) if b not in completed]
    timing={};actual_workers=1
    # Threadpoolctl is scoped around the whole pool; no thread changes global
    # BLAS limits independently of other active workers.
    with threadpool_limits(limits=1,user_api='blas'):
        if workers=='auto' and len(remaining)>=4 and (os.cpu_count() or 1)>=2:
            start=time.perf_counter();run_batch(remaining[:2],1);serial=time.perf_counter()-start
            start=time.perf_counter();run_batch(remaining[2:4],2);parallel=time.perf_counter()-start
            actual_workers=2 if parallel<.90*serial else 1
            timing=dict(serial_two_draws_seconds=serial,parallel_two_draws_seconds=parallel)
            remaining=remaining[4:]
            if actual_workers == 2 and len(remaining) >= 4 and (os.cpu_count() or 1) >= 4:
                start=time.perf_counter();run_batch(remaining[:4],4);four=time.perf_counter()-start
                timing['parallel_four_draws_seconds']=four
                if four/4 < .90*parallel/2:
                    actual_workers=4
                remaining=remaining[4:]
            print(f'Automatic worker calibration selected {actual_workers} worker(s).',flush=True)
        elif workers!='auto':actual_workers=workers
        run_batch(remaining,actual_workers)
    export_aggregates()
    rows=[]
    for name,(reference,flag) in references.items():
        ids=[b for b in sorted(completed) if name in completed[b]]
        reference.bootstrap_coefficients=np.asarray([completed[b][name] for b in ids],dtype=float).reshape(-1,len(names[name]))
        reference.bootstrap_inference_valid=bool(len(ids)>=required and not base_issues[name])
        if flag:reference.bootstrap_profiles=np.asarray([completed[b]['profiles'] for b in ids],dtype=float).reshape(-1,len(metadata),3)
        rows.append(dict(model=name,requested_draws=reps,successful_draws=len(ids),excluded_or_failed=reps-len(ids),
            minimum_successes_required=required,inference_valid=reference.bootstrap_inference_valid,
            original_fit_issues='; '.join(base_issues[name]),seed=BOOTSTRAP_SEED,
            method='nonparametric household cluster bootstrap',synthetic_smoke_test=RUN_MODE=='synthetic',
            worker_setting=str(workers),selected_workers=actual_workers,checkpoint_directory=str(checkpoint_dir),
            incidence_cache_mode=proposed.prepared.arrays._incidence_cache_mode,**timing))
        if not reference.bootstrap_inference_valid:warnings.warn(f'{name}: bootstrap inference withheld; inspect 04_bootstrap diagnostics.')
    summary=pd.DataFrame(rows);summary.to_csv(folder/'bootstrap_summary.csv',index=False)
    return summary


def full_sample_multistart(prepared, iterations, destination):
    if type(N_STARTS) is not int or N_STARTS < 2:
        raise ValueError('N_STARTS must be an integer of at least two.')
    incidence = fit_incidence_models(prepared.arrays)
    for fit in incidence.values():
        finalize_diagnostics(fit)
    selected, rows = [], []
    for model, switching in [('proposed_hmm', True), ('nested_no_switch', False)]:
        best_key, best, best_row = None, None, None
        def fit_start(start_id):
            # Recursion caches and fit diagnostics are private to this worker.
            local = copy(prepared)
            local.arrays = copy(prepared.arrays)
            result = None
            row = dict(model=model, start=start_id, start_pi=start_probability(start_id), selected=False)
            try:
                kwargs = dict(max_em_iters=iterations, start_id=start_id, verbose=False)
                if switching:
                    result = estimate_model(local, incidence=incidence, **kwargs)
                else:
                    result = estimate_no_switch_model(SimpleNamespace(prepared=local,incidence_models=incidence), **kwargs)
                issues = fit_acceptance(result, switching)
                ll = result.log_likelihood
                row.update(log_likelihood=ll, final_pi_before_alignment=result.pi_state2,
                           converged=result.converged, iterations=len(result.history_df),
                           issues='; '.join(issues), seconds=result.estimation_seconds)
            except (ValueError, FloatingPointError, np.linalg.LinAlgError) as exc:
                row.update(issues=str(exc))
            return row, result

        workers = min(START_WORKERS, N_STARTS, os.cpu_count() or 1)
        if workers < 1:
            raise ValueError('START_WORKERS must be positive.')
        with ThreadPoolExecutor(max_workers=workers) as pool:
            # Limit live fits to one batch. Deterministic ties go to the earlier start.
            for first in range(0, N_STARTS, workers):
                for row, result in pool.map(fit_start, range(first, min(first+workers, N_STARTS))):
                    rows.append(row)
                    ll = row.get('log_likelihood', np.nan)
                    if result is not None and np.isfinite(ll):
                        key = (not row['issues'], result.converged, ll)
                        if best_key is None or key > best_key:
                            best_key, best, best_row = key, result, row
                    print(f"{model} start {row['start']+1}/{N_STARTS}: LL={ll:.6f}, "
                          f"issues={row.get('issues') or 'none'}", flush=True)
        if best_row is not None:
            best_row['selected'] = True
        pd.DataFrame(rows).to_csv(destination/'multistart_diagnostics.csv', index=False)
        if best is None:
            raise ValueError(f'No finite {model} start. See multistart_diagnostics.csv.')
        canonicalize_state_labels(best, switching)
        selected.append(best)
    return tuple(selected)


## Execute and resume

Run all cells after editing the settings. Outputs go under `V42_Sugar_Choice_Bootstrap_Results_Optimized` in a subfolder named for the inclusive year window, minimum observed years and gap policy. Switching between filters preserves each sample’s checkpoints. Rerunning the same sample verifies the source, specification and original estimates before reusing completed draws.

Start with `sample_audit.csv`, `yogurt_code_audit.csv`, `multistart_diagnostics.csv` and `04_bootstrap/bootstrap_summary.csv`. The healthy-choice table is `02_proposed_hmm/11_posterior_healthy_choices.csv`. Inference requires at least 100 accepted draws and 80% of requested draws in full-estimation mode. The synthetic mode uses five draws for software testing.
The optimized run writes the same result filenames in a sibling folder. Its checkpoints resume across compatible optimized runs. Any saved cell output below predates this edit.


In [9]:
def make_synthetic_parquet(path, n_households=120, seed=SEED):
    """Artificial fixture with all schema fields and both states; never research data."""
    rng=np.random.default_rng(seed); rows=[]; trip=0
    for h in range(n_households):
        state=int(rng.random()<.3)
        for t in range(4):
            events=rng.binomial(1,.15,6)
            if t: state=int(rng.random()<(.8 if state else .2))
            controls=[rng.binomial(1,.5),rng.uniform(1,12),rng.integers(1,6),rng.uniform(25,75)]
            for j in range(12):
                trip+=1
                row=dict(hhid=f'{h:06d}',panel_year=2008+t,trip_id=str(trip),purchase_date=pd.Timestamp(2008+t,1,1)+pd.Timedelta(days=j*20),food_spend=float(rng.integers(10,100)),initial_fresh_fv_share_2007=.3,other_unemployment=0)
                row.update(zip(TRANSITION_EVENT_COLUMNS,events)); row.update(zip(GATE_COVARIATE_COLS,controls))
                for c,cfg in CATEGORY_CONFIG.items():
                    bought=rng.random()<.7
                    y=0
                    if bought:
                        y=1 if rng.random()<(.75 if state else .25) else (2 if c!='cereal' or rng.random()<.5 else 3)
                    row[c+'_y']=y
                    for v in cfg['spending']: row[v]=0.
                    if y: row[cfg['spending'][min(y-1,len(cfg['spending'])-1)]]=2.
                row['milk_y']=int(rng.choice([0,1,2],p=[.65,.15,.20]))
                milk_columns=['skimmed_milk_spending','low_fat_milk_spending','other_milk_spending']
                for v in milk_columns: row[v]=0.
                if row['milk_y']:
                    column=milk_columns[0] if row['milk_y']==1 else milk_columns[int(rng.integers(1,3))]
                    row[column]=2.
                rows.append(row)
    pd.DataFrame(rows).to_parquet(path,index=False)


def run_v42():
    started=time.perf_counter()
    tag=sample_tag()
    if RUN_MODE not in ('full_estimation','validate_only','synthetic'):
        raise ValueError('Unknown RUN_MODE')
    if RUN_MODE=='synthetic':
        root=Path.cwd()/'V42_Sugar_Choice_Synthetic_Results_Optimized'
        root.mkdir(parents=True,exist_ok=True)
        source=root/'synthetic_input.parquet'
        if not source.exists():
            make_synthetic_parquet(source)
        iterations=SYNTHETIC_EM_ITERS
    else:
        source=INPUT_PATH
        root=OUTPUT_DIR
        iterations=MAX_EM_ITERS
        if not source.is_file(): raise FileNotFoundError(source)
    destination=root/tag
    destination.mkdir(parents=True,exist_ok=True)
    try:
        prepared=prepare_parquet(source,destination/'duckdb_temp')
    except EmptySample as exc:
        pd.DataFrame([dict(status='sample unavailable',reason=str(exc),year_min=YEAR_MIN,year_max=YEAR_MAX,
                           minimum_observed_years=MIN_OBSERVED_YEARS_PER_HOUSEHOLD)]).to_csv(destination/'sample_audit.csv',index=False)
        print(f'{exc} See {destination}')
        return
    try:
        print(f'Preparation: {time.perf_counter()-started:.1f} seconds')
        print("Numerical engine:", "Numba" if USE_NUMBA and numba is not None else "NumPy")
        for name in ('beverage_merge_audit','yogurt_code_audit'):
            prepared.df.attrs[name].to_csv(destination/(name+'.csv'),index=False)
        pd.DataFrame([prepared.df.attrs['sorting_audit']]).to_csv(destination/'sample_audit.csv',index=False)
        if RUN_MODE=='validate_only':
            print(f'Validation finished. Results: {destination.resolve()}')
            return
        with threadpool_limits(limits=1,user_api='blas'):
            proposed,nested=full_sample_multistart(prepared,iterations,destination)
        run_household_bootstrap(proposed,nested,destination,
            reps=SYNTHETIC_BOOTSTRAP_REPS if RUN_MODE=='synthetic' else BOOTSTRAP_REPS,
            min_success=2 if RUN_MODE=='synthetic' else BOOTSTRAP_MIN_SUCCESS,
            min_fraction=0.0 if RUN_MODE=='synthetic' else BOOTSTRAP_MIN_SUCCESS_FRACTION)
        comparison=save_outputs(proposed,nested,destination)
        print(f'Total elapsed: {time.perf_counter()-started:.1f} seconds. Results: {destination.resolve()}')
        return proposed,nested,comparison
    finally:
        prepared.arrays._connection.close()


v42_results = run_v42()
if v42_results is not None:
    proposed, nested, comparison = v42_results


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Error: Traceback (most recent call last):

  File "c:\Users\rwen13\AppData\Local\anaconda3\Lib\site-packages\traitlets\traitlets.py", line 708, in __set__
    def __set__(self, obj: HasTraits, value: S) -> None:
    

KeyboardInterrupt
